# Tests for `time_series_tools`
This notebook dynamically imports the module and runs a basic test for each exported function so you can inspect outputs.

In [4]:
# 1) Import Libraries
import os, sys, types, inspect, importlib.util, contextlib, io, time, json, traceback, asyncio
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import pytz

In [5]:
# 2) Configure Source File and Target Functions
module_path = os.path.join(os.getcwd(), "time_series_tools.py")
assert os.path.exists(module_path), module_path
include_patterns = ["get_timezone","is_weekday_mask","is_market_hour_mask","is_market_hour_minute_mask","median_delta_time","find_large_gaps","chunks","find_gaps_and_chunks","find_blocks","align_chunks"]
exclude_patterns = []
explicit_functions = []  # set names to only test those

In [6]:
# 3) Dynamically Import the Module
spec = importlib.util.spec_from_file_location("time_series_tools", module_path)
mod = importlib.util.module_from_spec(spec)
loader = spec.loader
assert loader is not None
loader.exec_module(mod)
mod

<module 'time_series_tools' from 'c:\\Users\\khazy\\OneDrive\\Documents\\DCG_Release_Tests_py11\\time_series_tools.py'>

In [15]:
# 4) Discover Functions to Test
def want(name: str) -> bool:
    if explicit_functions and name not in explicit_functions:
        return False
    if include_patterns and name not in include_patterns:
        return False
    if name in exclude_patterns:
        return False
    return True

funcs = {name: fn for name, fn in inspect.getmembers(mod, inspect.isfunction) if want(name)}
sorted(funcs.keys())

['align_chunks',
 'chunks',
 'find_blocks',
 'find_gaps_and_chunks',
 'find_large_gaps',
 'is_market_hour_mask',
 'is_market_hour_minute_mask',
 'is_weekday_mask',
 'median_delta_time']

In [16]:
# 5) Define One Test Case per Function
def default_for_param(param: inspect.Parameter):
    if param.default is not inspect._empty:
        return param.default
    ann = param.annotation
    if ann in (int, np.int64):
        return 1
    if ann in (float, np.float64):
        return 1.0
    if ann is bool:
        return True
    if ann is str:
        return "sample"
    return None

# Sample data for functions
start_dt = datetime(2024, 8, 5, 0, 0)  # Monday
unix_times_weekly_hourly = np.array([int((start_dt + pd.Timedelta(hours=i)).timestamp()) for i in range(7*24)])
unix_times_minutely = np.array([int((start_dt + pd.Timedelta(minutes=i)).timestamp()) for i in range(24*60)])

# DataFrame with gaps for gap utilities
df_dates = pd.Series(unix_times_weekly_hourly.copy())
# Inject two artificial large gaps
df_dates.iloc[100] += 3600 * 5  # 5 hours jump
df_dates.iloc[150] += 3600 * 10 # 10 hours jump
df = pd.DataFrame({"Date": df_dates.astype(int)})
df['Time_Shift'] = 0

test_cases = {
    'get_timezone': {"args": ["US/Eastern"], "kwargs": {}},
    'is_weekday_mask': {"args": [unix_times_weekly_hourly], "kwargs": {"tz_str": "US/Eastern"}},
    'is_market_hour_mask': {"args": [unix_times_weekly_hourly], "kwargs": {"market_open": 9, "market_close": 16, "tz_str": "US/Eastern"}},
    'is_market_hour_minute_mask': {"args": [unix_times_minutely], "kwargs": {"market_open_hour": 9, "market_open_minute": 30, "market_close_hour": 16, "market_close_minute": 0, "tz_str": "US/Eastern"}},
    'median_delta_time': {"args": [df['Date']], "kwargs": {}},
    'find_large_gaps': {"args": [df['Date']], "kwargs": {}},
    'chunks': {"args": [[], int(df['Date'].min()), int(df['Date'].max())], "kwargs": {}},
    'find_gaps_and_chunks': {"args": [df['Date']], "kwargs": {}},
    'find_blocks': {"args": [(np.array([False]*10 + [True]*5 + [False]*3 + [True]*2))], "kwargs": {}},
    'align_chunks': {"args": [[{'Type':'chunk','start': int(df['Date'].iloc[0]), 'stop': int(df['Date'].iloc[50]), 'duration': int(df['Date'].iloc[50]-df['Date'].iloc[0])}], df], "kwargs": {}},
}

In [17]:
# 6) Execute Functions and Capture Outputs
results = []
for name, fn in funcs.items():
    cfg = test_cases.get(name, {"args": [], "kwargs": {}})
    args = cfg.get("args", [])
    kwargs = cfg.get("kwargs", {})
    stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
    start = time.perf_counter()
    error = None
    ret = None
    try:
        with contextlib.redirect_stdout(stdout_buf), contextlib.redirect_stderr(stderr_buf):
            if inspect.iscoroutinefunction(fn):
                ret = asyncio.run(fn(*args, **kwargs))
            else:
                ret = fn(*args, **kwargs)
    except Exception as e:
        error = traceback.format_exc()
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    results.append({
        "name": name,
        "args": args,
        "kwargs": kwargs,
        "return_type": None if ret is None else type(ret).__name__,
        "return_value": repr(ret)[:500],
        "stdout": stdout_buf.getvalue(),
        "stderr": stderr_buf.getvalue(),
        "elapsed_ms": round(elapsed_ms, 3),
        "error": error,
    })
len(results)

9

In [18]:
# 7) Display Results in a Table
try:
    import pandas as _pd
    _df = _pd.DataFrame(results)
    display(_df[["name","return_type","elapsed_ms","error"]])
except Exception:
    print(json.dumps(results, indent=2)[:4000])

,name,return_type,elapsed_ms,error
0,align_chunks,None,0.205,None
1,chunks,list,0.016,None
2,find_blocks,list,0.083,None
3,find_gaps_and_chunks,list,1.997,None
4,find_large_gaps,list,1.421,None
5,is_market_hour_mask,ndarray,50.357,None
6,is_market_hour_minute_mask,ndarray,9.269,None
7,is_weekday_mask,ndarray,0.857,None
8,median_delta_time,float64,0.966,None


In [19]:
# 8) Detailed Inspection per Function (Optional)
import pprint
for row in results:
    print("\n===", row["name"], "===")
    print("args:", row["args"])
    print("kwargs:", row["kwargs"])
    print("return_type:", row["return_type"])
    print("elapsed_ms:", row["elapsed_ms"])
    if row["error"]:
        print("ERROR:\n", row["error"])
    else:
        print("return_value (repr):")
        pprint.pprint(row["return_value"])
        if row["stdout"]:
            print("stdout:\n", row["stdout"][:1000])
        if row["stderr"]:
            print("stderr:\n", row["stderr"][:1000])


=== align_chunks ===
args: [[{'Type': 'chunk', 'start': 1722830400, 'stop': 1723010400, 'duration': 180000}],            Date  Time_Shift
0    1722830400           0
1    1722834000           0
2    1722837600           0
3    1722841200           0
4    1722844800           0
..          ...         ...
163  1723417200           0
164  1723420800           0
165  1723424400           0
166  1723428000           0
167  1723431600           0

[168 rows x 2 columns]]
kwargs: {}
return_type: None
elapsed_ms: 0.205
return_value (repr):
'None'

=== chunks ===
args: [[], 1722830400, 1723431600]
kwargs: {}
return_type: list
elapsed_ms: 0.016
return_value (repr):
("[{'Type': 'chunk', 'start': 1722830400, 'stop': 1723431600, 'duration': "
 '601200}]')

=== find_blocks ===
args: [array([False, False, False, False, False, False, False, False, False,
       False,  True,  True,  True,  True,  True, False, False, False,
        True,  True])]
kwargs: {}
return_type: list
elapsed_ms: 0.083
return_

# Test Return from Thunder Client

In [70]:
from datetime import datetime
import json
import polars as pl
import time, statistics
import calendar as _calendar


In [15]:
json_data = [
  {
    "date": "2025-09-19 15:55:00",
    "open": 245.44,
    "low": 245.39,
    "high": 245.62,
    "close": 245.43,
    "volume": 4157082
  },
  {
    "date": "2025-09-19 15:50:00",
    "open": 244.95,
    "low": 244.95,
    "high": 245.875,
    "close": 245.6,
    "volume": 3935279
  },
  {
    "date": "2025-09-19 15:45:00",
    "open": 244.8609,
    "low": 244.7646,
    "high": 245.08,
    "close": 244.95,
    "volume": 1193115
  },
  {
    "date": "2025-09-19 15:40:00",
    "open": 245.1,
    "low": 244.81,
    "high": 245.1608,
    "close": 244.8388,
    "volume": 995815
  },
  {
    "date": "2025-09-19 15:35:00",
    "open": 245.62,
    "low": 245.0211,
    "high": 245.6594,
    "close": 245.08,
    "volume": 1169350
  },
  {
    "date": "2025-09-19 15:30:00",
    "open": 245.635,
    "low": 245.5144,
    "high": 245.905,
    "close": 245.63,
    "volume": 1007029
  },
  {
    "date": "2025-09-19 15:25:00",
    "open": 245.7227,
    "low": 245.595,
    "high": 245.88,
    "close": 245.595,
    "volume": 955973
  },
  {
    "date": "2025-09-19 15:20:00",
    "open": 245.579,
    "low": 245.579,
    "high": 245.74,
    "close": 245.74,
    "volume": 544158
  },
  {
    "date": "2025-09-19 15:15:00",
    "open": 245.6926,
    "low": 245.48,
    "high": 245.6926,
    "close": 245.57,
    "volume": 502547
  },
  {
    "date": "2025-09-19 15:10:00",
    "open": 245.64,
    "low": 245.55,
    "high": 245.7688,
    "close": 245.6228,
    "volume": 616378
  },
  {
    "date": "2025-09-19 15:05:00",
    "open": 245.56,
    "low": 245.55,
    "high": 245.76,
    "close": 245.67,
    "volume": 1239522
  },
  {
    "date": "2025-09-19 15:00:00",
    "open": 245.484,
    "low": 245.4048,
    "high": 245.6712,
    "close": 245.56,
    "volume": 441683
  },
  {
    "date": "2025-09-19 14:55:00",
    "open": 245.42,
    "low": 245.2,
    "high": 245.543,
    "close": 245.5134,
    "volume": 597810
  },
  {
    "date": "2025-09-19 14:50:00",
    "open": 245.7288,
    "low": 245.44,
    "high": 245.7602,
    "close": 245.46,
    "volume": 548719
  },
  {
    "date": "2025-09-19 14:45:00",
    "open": 245.85,
    "low": 245.6625,
    "high": 245.9021,
    "close": 245.7163,
    "volume": 427970
  },
  {
    "date": "2025-09-19 14:40:00",
    "open": 245.9645,
    "low": 245.8,
    "high": 246,
    "close": 245.8119,
    "volume": 519018
  },
  {
    "date": "2025-09-19 14:35:00",
    "open": 245.6,
    "low": 245.6,
    "high": 245.72,
    "close": 245.63,
    "volume": 5941
  },
  {
    "date": "2025-09-19 14:30:00",
    "open": 245.48,
    "low": 245.35,
    "high": 245.81,
    "close": 245.61,
    "volume": 564462
  },
  {
    "date": "2025-09-19 14:25:00",
    "open": 245.68,
    "low": 245.41,
    "high": 245.69,
    "close": 245.49,
    "volume": 700996
  },
  {
    "date": "2025-09-19 14:20:00",
    "open": 245.9,
    "low": 245.63,
    "high": 246.06,
    "close": 245.69,
    "volume": 570363
  },
  {
    "date": "2025-09-19 14:15:00",
    "open": 246.12,
    "low": 245.74,
    "high": 246.26,
    "close": 245.88,
    "volume": 1908481
  },
  {
    "date": "2025-09-19 14:10:00",
    "open": 245.81,
    "low": 245.59,
    "high": 246.13,
    "close": 246.1,
    "volume": 907129
  },
  {
    "date": "2025-09-19 14:05:00",
    "open": 245.47,
    "low": 245.47,
    "high": 245.97,
    "close": 245.85,
    "volume": 1677860
  },
  {
    "date": "2025-09-19 14:00:00",
    "open": 245.41,
    "low": 244.96,
    "high": 245.63,
    "close": 245.47,
    "volume": 704379
  },
  {
    "date": "2025-09-19 13:55:00",
    "open": 245.08,
    "low": 245.03,
    "high": 245.86,
    "close": 245.39,
    "volume": 857948
  },
  {
    "date": "2025-09-19 13:50:00",
    "open": 245.44,
    "low": 245.06,
    "high": 245.47,
    "close": 245.12,
    "volume": 484555
  },
  {
    "date": "2025-09-19 13:45:00",
    "open": 245.35,
    "low": 245.29,
    "high": 245.5,
    "close": 245.43,
    "volume": 350287
  },
  {
    "date": "2025-09-19 13:40:00",
    "open": 245.35,
    "low": 245.22,
    "high": 245.52,
    "close": 245.37,
    "volume": 472873
  },
  {
    "date": "2025-09-19 13:35:00",
    "open": 244.93,
    "low": 244.93,
    "high": 245.47,
    "close": 245.4,
    "volume": 555017
  },
  {
    "date": "2025-09-19 13:30:00",
    "open": 245.24,
    "low": 244.91,
    "high": 245.3,
    "close": 244.93,
    "volume": 557079
  },
  {
    "date": "2025-09-19 13:25:00",
    "open": 245.19,
    "low": 245.05,
    "high": 245.42,
    "close": 245.24,
    "volume": 636278
  },
  {
    "date": "2025-09-19 13:20:00",
    "open": 245.09,
    "low": 245.04,
    "high": 245.25,
    "close": 245.19,
    "volume": 425375
  },
  {
    "date": "2025-09-19 13:15:00",
    "open": 245.19,
    "low": 245.03,
    "high": 245.31,
    "close": 245.1,
    "volume": 447650
  },
  {
    "date": "2025-09-19 13:10:00",
    "open": 245.29,
    "low": 245.11,
    "high": 245.5,
    "close": 245.18,
    "volume": 490475
  },
  {
    "date": "2025-09-19 13:05:00",
    "open": 245.02,
    "low": 245.01,
    "high": 245.42,
    "close": 245.3,
    "volume": 515114
  },
  {
    "date": "2025-09-19 13:00:00",
    "open": 245.16,
    "low": 244.9,
    "high": 245.4,
    "close": 245.03,
    "volume": 719380
  },
  {
    "date": "2025-09-19 12:55:00",
    "open": 245.42,
    "low": 245.12,
    "high": 245.42,
    "close": 245.17,
    "volume": 596059
  },
  {
    "date": "2025-09-19 12:50:00",
    "open": 245.48,
    "low": 245.11,
    "high": 245.55,
    "close": 245.44,
    "volume": 935924
  },
  {
    "date": "2025-09-19 12:45:00",
    "open": 245.54,
    "low": 245.44,
    "high": 245.92,
    "close": 245.48,
    "volume": 2715743
  },
  {
    "date": "2025-09-19 12:40:00",
    "open": 244.82,
    "low": 244.8,
    "high": 245.8,
    "close": 245.55,
    "volume": 1474934
  },
  {
    "date": "2025-09-19 12:35:00",
    "open": 244.66,
    "low": 244.66,
    "high": 245.12,
    "close": 244.83,
    "volume": 1512063
  },
  {
    "date": "2025-09-19 12:30:00",
    "open": 244.51,
    "low": 244.27,
    "high": 244.71,
    "close": 244.68,
    "volume": 1800837
  },
  {
    "date": "2025-09-19 12:25:00",
    "open": 244.29,
    "low": 244.23,
    "high": 244.57,
    "close": 244.52,
    "volume": 566682
  },
  {
    "date": "2025-09-19 12:20:00",
    "open": 244.08,
    "low": 244.03,
    "high": 244.49,
    "close": 244.29,
    "volume": 878050
  },
  {
    "date": "2025-09-19 12:15:00",
    "open": 244.28,
    "low": 243.83,
    "high": 244.39,
    "close": 244.08,
    "volume": 2142619
  },
  {
    "date": "2025-09-19 12:10:00",
    "open": 243.65,
    "low": 243.65,
    "high": 244.27,
    "close": 244.26,
    "volume": 1155047
  },
  {
    "date": "2025-09-19 12:05:00",
    "open": 243.39,
    "low": 243.38,
    "high": 243.67,
    "close": 243.63,
    "volume": 483963
  },
  {
    "date": "2025-09-19 12:00:00",
    "open": 243.27,
    "low": 243.18,
    "high": 243.45,
    "close": 243.39,
    "volume": 575563
  },
  {
    "date": "2025-09-19 11:55:00",
    "open": 242.91,
    "low": 242.65,
    "high": 243.39,
    "close": 243.28,
    "volume": 738286
  },
  {
    "date": "2025-09-19 11:50:00",
    "open": 242.61,
    "low": 242.58,
    "high": 242.94,
    "close": 242.92,
    "volume": 407150
  },
  {
    "date": "2025-09-19 11:45:00",
    "open": 242.64,
    "low": 242.33,
    "high": 242.7,
    "close": 242.6,
    "volume": 380479
  },
  {
    "date": "2025-09-19 11:40:00",
    "open": 242.41,
    "low": 242.35,
    "high": 242.87,
    "close": 242.64,
    "volume": 503319
  },
  {
    "date": "2025-09-19 11:35:00",
    "open": 242.78,
    "low": 242.34,
    "high": 242.85,
    "close": 242.39,
    "volume": 497574
  },
  {
    "date": "2025-09-19 11:30:00",
    "open": 242.26,
    "low": 242.04,
    "high": 242.79,
    "close": 242.79,
    "volume": 702373
  },
  {
    "date": "2025-09-19 11:25:00",
    "open": 241.89,
    "low": 241.83,
    "high": 242.29,
    "close": 242.27,
    "volume": 472507
  },
  {
    "date": "2025-09-19 11:20:00",
    "open": 242.09,
    "low": 241.8,
    "high": 242.26,
    "close": 241.91,
    "volume": 704808
  },
  {
    "date": "2025-09-19 11:15:00",
    "open": 242.2,
    "low": 241.99,
    "high": 242.22,
    "close": 242.09,
    "volume": 655070
  },
  {
    "date": "2025-09-19 11:10:00",
    "open": 242.65,
    "low": 242.17,
    "high": 242.88,
    "close": 242.21,
    "volume": 591020
  },
  {
    "date": "2025-09-19 11:05:00",
    "open": 242.64,
    "low": 242.35,
    "high": 243,
    "close": 242.65,
    "volume": 937376
  },
  {
    "date": "2025-09-19 11:00:00",
    "open": 242.87,
    "low": 242.56,
    "high": 243,
    "close": 242.63,
    "volume": 690080
  },
  {
    "date": "2025-09-19 10:55:00",
    "open": 243.1,
    "low": 242.54,
    "high": 243.16,
    "close": 242.83,
    "volume": 1173549
  },
  {
    "date": "2025-09-19 10:50:00",
    "open": 242.23,
    "low": 242.23,
    "high": 243.15,
    "close": 243.08,
    "volume": 1675104
  },
  {
    "date": "2025-09-19 10:45:00",
    "open": 241.93,
    "low": 241.82,
    "high": 242.22,
    "close": 242.22,
    "volume": 807753
  },
  {
    "date": "2025-09-19 10:40:00",
    "open": 241.72,
    "low": 241.48,
    "high": 242.08,
    "close": 241.92,
    "volume": 1078279
  },
  {
    "date": "2025-09-19 10:35:00",
    "open": 241.7,
    "low": 241.4,
    "high": 241.83,
    "close": 241.72,
    "volume": 657640
  },
  {
    "date": "2025-09-19 10:30:00",
    "open": 241.15,
    "low": 241.06,
    "high": 241.7,
    "close": 241.7,
    "volume": 700172
  },
  {
    "date": "2025-09-19 10:25:00",
    "open": 240.83,
    "low": 240.8,
    "high": 241.23,
    "close": 241.15,
    "volume": 641270
  },
  {
    "date": "2025-09-19 10:20:00",
    "open": 240.55,
    "low": 240.45,
    "high": 241.09,
    "close": 240.84,
    "volume": 686856
  },
  {
    "date": "2025-09-19 10:15:00",
    "open": 241.32,
    "low": 240.24,
    "high": 241.36,
    "close": 240.54,
    "volume": 958727
  },
  {
    "date": "2025-09-19 10:10:00",
    "open": 240.94,
    "low": 240.9,
    "high": 241.51,
    "close": 241.33,
    "volume": 957249
  },
  {
    "date": "2025-09-19 10:05:00",
    "open": 240.82,
    "low": 240.55,
    "high": 240.95,
    "close": 240.95,
    "volume": 613901
  },
  {
    "date": "2025-09-19 10:00:00",
    "open": 240.9,
    "low": 240.62,
    "high": 241.3,
    "close": 240.79,
    "volume": 867006
  },
  {
    "date": "2025-09-19 09:55:00",
    "open": 241.24,
    "low": 240.7,
    "high": 241.31,
    "close": 240.87,
    "volume": 703661
  },
  {
    "date": "2025-09-19 09:50:00",
    "open": 241.26,
    "low": 240.65,
    "high": 241.73,
    "close": 241.23,
    "volume": 1097601
  },
  {
    "date": "2025-09-19 09:45:00",
    "open": 240.95,
    "low": 240.36,
    "high": 241.3,
    "close": 241.28,
    "volume": 3672029
  },
  {
    "date": "2025-09-19 09:40:00",
    "open": 241.26,
    "low": 240.44,
    "high": 241.35,
    "close": 240.94,
    "volume": 1313098
  },
  {
    "date": "2025-09-19 09:35:00",
    "open": 241.69,
    "low": 240.8,
    "high": 242.33,
    "close": 241.23,
    "volume": 1817413
  },
  {
    "date": "2025-09-19 09:30:00",
    "open": 241.18,
    "low": 240.98,
    "high": 242.7,
    "close": 241.7,
    "volume": 16075909
  },
  {
    "date": "2025-09-18 15:55:00",
    "open": 237.89,
    "low": 237.74,
    "high": 238.11,
    "close": 237.87,
    "volume": 1802623
  },
  {
    "date": "2025-09-18 15:50:00",
    "open": 237.58,
    "low": 237.44,
    "high": 237.96,
    "close": 237.88,
    "volume": 849856
  },
  {
    "date": "2025-09-18 15:45:00",
    "open": 237.79,
    "low": 237.55,
    "high": 237.8,
    "close": 237.62,
    "volume": 333690
  },
  {
    "date": "2025-09-18 15:40:00",
    "open": 237.83,
    "low": 237.51,
    "high": 237.91,
    "close": 237.79,
    "volume": 400899
  },
  {
    "date": "2025-09-18 15:35:00",
    "open": 237.69,
    "low": 237.68,
    "high": 237.9,
    "close": 237.82,
    "volume": 354680
  },
  {
    "date": "2025-09-18 15:30:00",
    "open": 237.46,
    "low": 237.44,
    "high": 237.7,
    "close": 237.68,
    "volume": 307635
  },
  {
    "date": "2025-09-18 15:25:00",
    "open": 237.39,
    "low": 237.26,
    "high": 237.51,
    "close": 237.45,
    "volume": 347341
  },
  {
    "date": "2025-09-18 15:20:00",
    "open": 237.57,
    "low": 237.31,
    "high": 237.61,
    "close": 237.33,
    "volume": 504051
  },
  {
    "date": "2025-09-18 15:15:00",
    "open": 237.61,
    "low": 237.51,
    "high": 237.73,
    "close": 237.53,
    "volume": 388825
  },
  {
    "date": "2025-09-18 15:10:00",
    "open": 237.56,
    "low": 237.52,
    "high": 237.65,
    "close": 237.62,
    "volume": 329625
  },
  {
    "date": "2025-09-18 15:05:00",
    "open": 237.76,
    "low": 237.52,
    "high": 237.77,
    "close": 237.57,
    "volume": 274658
  },
  {
    "date": "2025-09-18 15:00:00",
    "open": 237.69,
    "low": 237.66,
    "high": 237.79,
    "close": 237.76,
    "volume": 213763
  },
  {
    "date": "2025-09-18 14:55:00",
    "open": 237.67,
    "low": 237.66,
    "high": 237.8,
    "close": 237.69,
    "volume": 315051
  },
  {
    "date": "2025-09-18 14:50:00",
    "open": 237.62,
    "low": 237.57,
    "high": 237.72,
    "close": 237.67,
    "volume": 254366
  },
  {
    "date": "2025-09-18 14:45:00",
    "open": 237.71,
    "low": 237.62,
    "high": 237.75,
    "close": 237.62,
    "volume": 207173
  },
  {
    "date": "2025-09-18 14:40:00",
    "open": 237.79,
    "low": 237.67,
    "high": 237.85,
    "close": 237.71,
    "volume": 240363
  },
  {
    "date": "2025-09-18 14:35:00",
    "open": 237.83,
    "low": 237.69,
    "high": 237.9,
    "close": 237.77,
    "volume": 251399
  },
  {
    "date": "2025-09-18 14:30:00",
    "open": 237.81,
    "low": 237.8,
    "high": 237.94,
    "close": 237.83,
    "volume": 214086
  },
  {
    "date": "2025-09-18 14:25:00",
    "open": 237.78,
    "low": 237.75,
    "high": 237.95,
    "close": 237.8,
    "volume": 220976
  },
  {
    "date": "2025-09-18 14:20:00",
    "open": 237.72,
    "low": 237.65,
    "high": 237.81,
    "close": 237.77,
    "volume": 204598
  },
  {
    "date": "2025-09-18 14:15:00",
    "open": 237.88,
    "low": 237.71,
    "high": 238.02,
    "close": 237.73,
    "volume": 236280
  },
  {
    "date": "2025-09-18 14:10:00",
    "open": 237.86,
    "low": 237.75,
    "high": 237.98,
    "close": 237.88,
    "volume": 331207
  },
  {
    "date": "2025-09-18 14:05:00",
    "open": 237.59,
    "low": 237.59,
    "high": 237.87,
    "close": 237.87,
    "volume": 307908
  },
  {
    "date": "2025-09-18 14:00:00",
    "open": 237.39,
    "low": 237.32,
    "high": 237.67,
    "close": 237.6,
    "volume": 259695
  },
  {
    "date": "2025-09-18 13:55:00",
    "open": 237.42,
    "low": 237.22,
    "high": 237.47,
    "close": 237.39,
    "volume": 279722
  },
  {
    "date": "2025-09-18 13:50:00",
    "open": 237.43,
    "low": 237.32,
    "high": 237.5,
    "close": 237.41,
    "volume": 220576
  },
  {
    "date": "2025-09-18 13:45:00",
    "open": 237.52,
    "low": 237.37,
    "high": 237.6,
    "close": 237.41,
    "volume": 188531
  },
  {
    "date": "2025-09-18 13:40:00",
    "open": 238,
    "low": 237.5,
    "high": 238,
    "close": 237.53,
    "volume": 256688
  },
  {
    "date": "2025-09-18 13:35:00",
    "open": 237.98,
    "low": 237.69,
    "high": 238.02,
    "close": 238.02,
    "volume": 306802
  },
  {
    "date": "2025-09-18 13:30:00",
    "open": 237.86,
    "low": 237.86,
    "high": 238.1,
    "close": 237.97,
    "volume": 197305
  },
  {
    "date": "2025-09-18 13:25:00",
    "open": 237.78,
    "low": 237.76,
    "high": 237.98,
    "close": 237.88,
    "volume": 233350
  },
  {
    "date": "2025-09-18 13:20:00",
    "open": 237.96,
    "low": 237.77,
    "high": 238.03,
    "close": 237.77,
    "volume": 763737
  },
  {
    "date": "2025-09-18 13:15:00",
    "open": 238.01,
    "low": 237.94,
    "high": 238.12,
    "close": 237.96,
    "volume": 258684
  },
  {
    "date": "2025-09-18 13:10:00",
    "open": 238,
    "low": 237.92,
    "high": 238.21,
    "close": 238.01,
    "volume": 240261
  },
  {
    "date": "2025-09-18 13:05:00",
    "open": 237.87,
    "low": 237.85,
    "high": 238.12,
    "close": 238.01,
    "volume": 283473
  },
  {
    "date": "2025-09-18 13:00:00",
    "open": 237.96,
    "low": 237.84,
    "high": 238.05,
    "close": 237.87,
    "volume": 219752
  },
  {
    "date": "2025-09-18 12:55:00",
    "open": 238,
    "low": 237.87,
    "high": 238.05,
    "close": 237.96,
    "volume": 227421
  },
  {
    "date": "2025-09-18 12:50:00",
    "open": 238.1,
    "low": 237.91,
    "high": 238.1,
    "close": 238.03,
    "volume": 208201
  },
  {
    "date": "2025-09-18 12:45:00",
    "open": 238.08,
    "low": 237.99,
    "high": 238.13,
    "close": 238.09,
    "volume": 326438
  },
  {
    "date": "2025-09-18 12:40:00",
    "open": 237.97,
    "low": 237.95,
    "high": 238.11,
    "close": 238.07,
    "volume": 207082
  },
  {
    "date": "2025-09-18 12:35:00",
    "open": 237.66,
    "low": 237.64,
    "high": 237.99,
    "close": 237.94,
    "volume": 280426
  },
  {
    "date": "2025-09-18 12:30:00",
    "open": 237.95,
    "low": 237.66,
    "high": 237.98,
    "close": 237.67,
    "volume": 307283
  },
  {
    "date": "2025-09-18 12:25:00",
    "open": 237.94,
    "low": 237.91,
    "high": 238.04,
    "close": 237.95,
    "volume": 170880
  },
  {
    "date": "2025-09-18 12:20:00",
    "open": 238.06,
    "low": 237.91,
    "high": 238.1,
    "close": 237.94,
    "volume": 199109
  },
  {
    "date": "2025-09-18 12:15:00",
    "open": 237.77,
    "low": 237.73,
    "high": 238.17,
    "close": 238.05,
    "volume": 393842
  },
  {
    "date": "2025-09-18 12:10:00",
    "open": 237.57,
    "low": 237.45,
    "high": 237.8,
    "close": 237.77,
    "volume": 270620
  },
  {
    "date": "2025-09-18 12:05:00",
    "open": 237.69,
    "low": 237.57,
    "high": 237.82,
    "close": 237.58,
    "volume": 235773
  },
  {
    "date": "2025-09-18 12:00:00",
    "open": 237.59,
    "low": 237.52,
    "high": 237.77,
    "close": 237.68,
    "volume": 296089
  },
  {
    "date": "2025-09-18 11:55:00",
    "open": 237.67,
    "low": 237.35,
    "high": 237.69,
    "close": 237.61,
    "volume": 389250
  },
  {
    "date": "2025-09-18 11:50:00",
    "open": 238.22,
    "low": 237.63,
    "high": 238.23,
    "close": 237.65,
    "volume": 452159
  },
  {
    "date": "2025-09-18 11:45:00",
    "open": 238.33,
    "low": 238.09,
    "high": 238.46,
    "close": 238.22,
    "volume": 301004
  },
  {
    "date": "2025-09-18 11:40:00",
    "open": 238.34,
    "low": 238.24,
    "high": 238.54,
    "close": 238.33,
    "volume": 450247
  },
  {
    "date": "2025-09-18 11:35:00",
    "open": 238.34,
    "low": 238.31,
    "high": 238.42,
    "close": 238.36,
    "volume": 232788
  },
  {
    "date": "2025-09-18 11:30:00",
    "open": 238.2,
    "low": 238.2,
    "high": 238.44,
    "close": 238.34,
    "volume": 217382
  },
  {
    "date": "2025-09-18 11:25:00",
    "open": 238.42,
    "low": 238.11,
    "high": 238.48,
    "close": 238.19,
    "volume": 241510
  },
  {
    "date": "2025-09-18 11:20:00",
    "open": 238.43,
    "low": 238.32,
    "high": 238.57,
    "close": 238.42,
    "volume": 224922
  },
  {
    "date": "2025-09-18 11:15:00",
    "open": 238.4,
    "low": 238.13,
    "high": 238.46,
    "close": 238.42,
    "volume": 328153
  },
  {
    "date": "2025-09-18 11:10:00",
    "open": 238.6,
    "low": 238.31,
    "high": 238.61,
    "close": 238.38,
    "volume": 245095
  },
  {
    "date": "2025-09-18 11:05:00",
    "open": 238.66,
    "low": 238.39,
    "high": 238.71,
    "close": 238.59,
    "volume": 363483
  },
  {
    "date": "2025-09-18 11:00:00",
    "open": 238.26,
    "low": 238.23,
    "high": 238.79,
    "close": 238.65,
    "volume": 508017
  },
  {
    "date": "2025-09-18 10:55:00",
    "open": 238.16,
    "low": 238.09,
    "high": 238.3,
    "close": 238.24,
    "volume": 317756
  },
  {
    "date": "2025-09-18 10:50:00",
    "open": 238.23,
    "low": 238.04,
    "high": 238.38,
    "close": 238.17,
    "volume": 416355
  },
  {
    "date": "2025-09-18 10:45:00",
    "open": 237.9,
    "low": 237.83,
    "high": 238.27,
    "close": 238.23,
    "volume": 425621
  },
  {
    "date": "2025-09-18 10:40:00",
    "open": 237.76,
    "low": 237.69,
    "high": 238.14,
    "close": 237.95,
    "volume": 480872
  },
  {
    "date": "2025-09-18 10:35:00",
    "open": 237.41,
    "low": 237.41,
    "high": 237.8,
    "close": 237.76,
    "volume": 474160
  },
  {
    "date": "2025-09-18 10:30:00",
    "open": 237.55,
    "low": 237.32,
    "high": 237.78,
    "close": 237.4,
    "volume": 561955
  },
  {
    "date": "2025-09-18 10:25:00",
    "open": 237.2,
    "low": 237.2,
    "high": 237.62,
    "close": 237.56,
    "volume": 630476
  },
  {
    "date": "2025-09-18 10:20:00",
    "open": 237.27,
    "low": 236.98,
    "high": 237.38,
    "close": 237.21,
    "volume": 689822
  },
  {
    "date": "2025-09-18 10:15:00",
    "open": 237.01,
    "low": 236.78,
    "high": 237.55,
    "close": 237.26,
    "volume": 737884
  },
  {
    "date": "2025-09-18 10:10:00",
    "open": 237.71,
    "low": 236.65,
    "high": 237.83,
    "close": 237.03,
    "volume": 1108219
  },
  {
    "date": "2025-09-18 10:05:00",
    "open": 238.11,
    "low": 237.62,
    "high": 238.31,
    "close": 237.7,
    "volume": 637121
  },
  {
    "date": "2025-09-18 10:00:00",
    "open": 238.42,
    "low": 237.88,
    "high": 238.42,
    "close": 238.14,
    "volume": 604153
  },
  {
    "date": "2025-09-18 09:55:00",
    "open": 238.06,
    "low": 237.92,
    "high": 238.59,
    "close": 238.41,
    "volume": 643610
  },
  {
    "date": "2025-09-18 09:50:00",
    "open": 238.05,
    "low": 237.92,
    "high": 238.41,
    "close": 238.06,
    "volume": 633874
  },
  {
    "date": "2025-09-18 09:45:00",
    "open": 238.1,
    "low": 237.63,
    "high": 238.34,
    "close": 238.05,
    "volume": 902126
  },
  {
    "date": "2025-09-18 09:40:00",
    "open": 239.47,
    "low": 238.07,
    "high": 239.7,
    "close": 238.08,
    "volume": 842559
  },
  {
    "date": "2025-09-18 09:35:00",
    "open": 239.19,
    "low": 238.96,
    "high": 239.77,
    "close": 239.5,
    "volume": 876903
  },
  {
    "date": "2025-09-18 09:30:00",
    "open": 239.97,
    "low": 239.12,
    "high": 241.18,
    "close": 239.21,
    "volume": 4077118
  },
  {
    "date": "2025-09-17 15:55:00",
    "open": 238.71,
    "low": 238.65,
    "high": 239.02,
    "close": 239.02,
    "volume": 1787645
  },
  {
    "date": "2025-09-17 15:50:00",
    "open": 238.84,
    "low": 238.48,
    "high": 238.95,
    "close": 238.71,
    "volume": 699569
  },
  {
    "date": "2025-09-17 15:45:00",
    "open": 238.94,
    "low": 238.67,
    "high": 239.02,
    "close": 238.83,
    "volume": 429087
  },
  {
    "date": "2025-09-17 15:40:00",
    "open": 239.05,
    "low": 238.83,
    "high": 239.11,
    "close": 238.97,
    "volume": 442298
  },
  {
    "date": "2025-09-17 15:35:00",
    "open": 238.92,
    "low": 238.89,
    "high": 239.24,
    "close": 239.03,
    "volume": 27853
  },
  {
    "date": "2025-09-17 15:30:00",
    "open": 239.26,
    "low": 238.78,
    "high": 239.28,
    "close": 238.91,
    "volume": 446639
  },
  {
    "date": "2025-09-17 15:25:00",
    "open": 239.17,
    "low": 239.01,
    "high": 239.39,
    "close": 239.28,
    "volume": 379945
  },
  {
    "date": "2025-09-17 15:20:00",
    "open": 239,
    "low": 238.9,
    "high": 239.31,
    "close": 239.14,
    "volume": 388833
  },
  {
    "date": "2025-09-17 15:15:00",
    "open": 238.87,
    "low": 238.83,
    "high": 239.28,
    "close": 239.04,
    "volume": 471768
  },
  {
    "date": "2025-09-17 15:10:00",
    "open": 239.27,
    "low": 238.74,
    "high": 239.46,
    "close": 238.84,
    "volume": 462012
  },
  {
    "date": "2025-09-17 15:05:00",
    "open": 238.97,
    "low": 238.85,
    "high": 239.37,
    "close": 239.27,
    "volume": 494506
  },
  {
    "date": "2025-09-17 15:00:00",
    "open": 238.58,
    "low": 238.52,
    "high": 239.03,
    "close": 238.95,
    "volume": 477725
  },
  {
    "date": "2025-09-17 14:55:00",
    "open": 237.92,
    "low": 237.9,
    "high": 238.63,
    "close": 238.56,
    "volume": 538855
  },
  {
    "date": "2025-09-17 14:50:00",
    "open": 238.29,
    "low": 237.74,
    "high": 238.37,
    "close": 237.92,
    "volume": 561808
  },
  {
    "date": "2025-09-17 14:45:00",
    "open": 238.64,
    "low": 237.9,
    "high": 238.64,
    "close": 238.3,
    "volume": 609500
  },
  {
    "date": "2025-09-17 14:40:00",
    "open": 238.59,
    "low": 238.34,
    "high": 239.05,
    "close": 238.56,
    "volume": 623195
  },
  {
    "date": "2025-09-17 14:35:00",
    "open": 239.09,
    "low": 238.57,
    "high": 239.24,
    "close": 238.59,
    "volume": 527314
  },
  {
    "date": "2025-09-17 14:30:00",
    "open": 239.57,
    "low": 239.05,
    "high": 239.62,
    "close": 239.11,
    "volume": 374758
  },
  {
    "date": "2025-09-17 14:25:00",
    "open": 239.27,
    "low": 239.27,
    "high": 239.6,
    "close": 239.55,
    "volume": 328670
  },
  {
    "date": "2025-09-17 14:20:00",
    "open": 238.51,
    "low": 238.45,
    "high": 239.34,
    "close": 239.24,
    "volume": 483970
  },
  {
    "date": "2025-09-17 14:15:00",
    "open": 238.85,
    "low": 238.54,
    "high": 238.9,
    "close": 238.54,
    "volume": 528082
  },
  {
    "date": "2025-09-17 14:10:00",
    "open": 239.47,
    "low": 238.88,
    "high": 239.58,
    "close": 238.89,
    "volume": 524706
  },
  {
    "date": "2025-09-17 14:05:00",
    "open": 239.55,
    "low": 239.38,
    "high": 239.67,
    "close": 239.45,
    "volume": 385093
  },
  {
    "date": "2025-09-17 14:00:00",
    "open": 239.43,
    "low": 239.21,
    "high": 239.83,
    "close": 239.53,
    "volume": 529302
  },
  {
    "date": "2025-09-17 13:55:00",
    "open": 239,
    "low": 238.95,
    "high": 239.47,
    "close": 239.44,
    "volume": 283386
  },
  {
    "date": "2025-09-17 13:50:00",
    "open": 239.1,
    "low": 238.92,
    "high": 239.27,
    "close": 238.98,
    "volume": 351711
  },
  {
    "date": "2025-09-17 13:45:00",
    "open": 239,
    "low": 238.99,
    "high": 239.21,
    "close": 239.12,
    "volume": 271718
  },
  {
    "date": "2025-09-17 13:40:00",
    "open": 239.32,
    "low": 238.97,
    "high": 239.36,
    "close": 239,
    "volume": 487308
  },
  {
    "date": "2025-09-17 13:35:00",
    "open": 239.55,
    "low": 239.33,
    "high": 239.6,
    "close": 239.33,
    "volume": 207845
  },
  {
    "date": "2025-09-17 13:30:00",
    "open": 239.52,
    "low": 239.48,
    "high": 239.62,
    "close": 239.57,
    "volume": 188258
  },
  {
    "date": "2025-09-17 13:25:00",
    "open": 239.47,
    "low": 239.4,
    "high": 239.57,
    "close": 239.53,
    "volume": 199672
  },
  {
    "date": "2025-09-17 13:20:00",
    "open": 239.57,
    "low": 239.46,
    "high": 239.6,
    "close": 239.48,
    "volume": 197749
  },
  {
    "date": "2025-09-17 13:15:00",
    "open": 239.5,
    "low": 239.46,
    "high": 239.56,
    "close": 239.56,
    "volume": 153442
  },
  {
    "date": "2025-09-17 13:10:00",
    "open": 239.57,
    "low": 239.47,
    "high": 239.58,
    "close": 239.5,
    "volume": 193805
  },
  {
    "date": "2025-09-17 13:05:00",
    "open": 239.56,
    "low": 239.5,
    "high": 239.61,
    "close": 239.57,
    "volume": 171211
  },
  {
    "date": "2025-09-17 13:00:00",
    "open": 239.57,
    "low": 239.5,
    "high": 239.61,
    "close": 239.55,
    "volume": 182367
  },
  {
    "date": "2025-09-17 12:55:00",
    "open": 239.45,
    "low": 239.44,
    "high": 239.6,
    "close": 239.59,
    "volume": 445477
  },
  {
    "date": "2025-09-17 12:50:00",
    "open": 239.51,
    "low": 239.41,
    "high": 239.7,
    "close": 239.44,
    "volume": 254447
  },
  {
    "date": "2025-09-17 12:45:00",
    "open": 239.51,
    "low": 239.36,
    "high": 239.54,
    "close": 239.5,
    "volume": 229689
  },
  {
    "date": "2025-09-17 12:40:00",
    "open": 239.61,
    "low": 239.41,
    "high": 239.62,
    "close": 239.53,
    "volume": 304149
  },
  {
    "date": "2025-09-17 12:35:00",
    "open": 239.63,
    "low": 239.59,
    "high": 239.77,
    "close": 239.62,
    "volume": 200142
  },
  {
    "date": "2025-09-17 12:30:00",
    "open": 239.68,
    "low": 239.61,
    "high": 239.77,
    "close": 239.63,
    "volume": 360473
  },
  {
    "date": "2025-09-17 12:25:00",
    "open": 239.56,
    "low": 239.52,
    "high": 239.69,
    "close": 239.68,
    "volume": 199921
  },
  {
    "date": "2025-09-17 12:20:00",
    "open": 239.53,
    "low": 239.5,
    "high": 239.59,
    "close": 239.54,
    "volume": 245728
  },
  {
    "date": "2025-09-17 12:15:00",
    "open": 239.51,
    "low": 239.5,
    "high": 239.66,
    "close": 239.53,
    "volume": 276525
  },
  {
    "date": "2025-09-17 12:10:00",
    "open": 239.38,
    "low": 239.29,
    "high": 239.5,
    "close": 239.5,
    "volume": 177003
  },
  {
    "date": "2025-09-17 12:05:00",
    "open": 239.36,
    "low": 239.28,
    "high": 239.47,
    "close": 239.38,
    "volume": 246661
  },
  {
    "date": "2025-09-17 12:00:00",
    "open": 239.57,
    "low": 239.31,
    "high": 239.62,
    "close": 239.35,
    "volume": 237742
  },
  {
    "date": "2025-09-17 11:55:00",
    "open": 239.45,
    "low": 239.44,
    "high": 239.69,
    "close": 239.58,
    "volume": 348729
  },
  {
    "date": "2025-09-17 11:50:00",
    "open": 239.52,
    "low": 239.39,
    "high": 239.52,
    "close": 239.46,
    "volume": 246056
  },
  {
    "date": "2025-09-17 11:45:00",
    "open": 239.55,
    "low": 239.48,
    "high": 239.6,
    "close": 239.52,
    "volume": 500843
  },
  {
    "date": "2025-09-17 11:40:00",
    "open": 239.5,
    "low": 239.35,
    "high": 239.55,
    "close": 239.55,
    "volume": 215286
  },
  {
    "date": "2025-09-17 11:35:00",
    "open": 239.72,
    "low": 239.45,
    "high": 239.73,
    "close": 239.51,
    "volume": 208249
  },
  {
    "date": "2025-09-17 11:30:00",
    "open": 239.47,
    "low": 239.43,
    "high": 239.78,
    "close": 239.73,
    "volume": 267928
  },
  {
    "date": "2025-09-17 11:25:00",
    "open": 239.53,
    "low": 239.41,
    "high": 239.61,
    "close": 239.49,
    "volume": 262414
  },
  {
    "date": "2025-09-17 11:20:00",
    "open": 239.36,
    "low": 239.25,
    "high": 239.58,
    "close": 239.52,
    "volume": 385438
  },
  {
    "date": "2025-09-17 11:15:00",
    "open": 239.34,
    "low": 239.2,
    "high": 239.47,
    "close": 239.36,
    "volume": 275844
  },
  {
    "date": "2025-09-17 11:10:00",
    "open": 239.52,
    "low": 239.25,
    "high": 239.52,
    "close": 239.35,
    "volume": 286941
  },
  {
    "date": "2025-09-17 11:05:00",
    "open": 239.52,
    "low": 239.45,
    "high": 239.58,
    "close": 239.53,
    "volume": 315037
  },
  {
    "date": "2025-09-17 11:00:00",
    "open": 239.54,
    "low": 239.32,
    "high": 239.62,
    "close": 239.51,
    "volume": 315075
  },
  {
    "date": "2025-09-17 10:55:00",
    "open": 239.95,
    "low": 239.53,
    "high": 240.01,
    "close": 239.53,
    "volume": 405603
  },
  {
    "date": "2025-09-17 10:50:00",
    "open": 239.67,
    "low": 239.58,
    "high": 239.99,
    "close": 239.95,
    "volume": 478036
  },
  {
    "date": "2025-09-17 10:45:00",
    "open": 239.47,
    "low": 239.46,
    "high": 239.67,
    "close": 239.66,
    "volume": 284142
  },
  {
    "date": "2025-09-17 10:40:00",
    "open": 239.59,
    "low": 239.39,
    "high": 239.61,
    "close": 239.48,
    "volume": 255430
  },
  {
    "date": "2025-09-17 10:35:00",
    "open": 239.56,
    "low": 239.45,
    "high": 239.74,
    "close": 239.6,
    "volume": 382462
  },
  {
    "date": "2025-09-17 10:30:00",
    "open": 239.63,
    "low": 239.53,
    "high": 239.83,
    "close": 239.56,
    "volume": 355888
  },
  {
    "date": "2025-09-17 10:25:00",
    "open": 239.75,
    "low": 239.61,
    "high": 239.96,
    "close": 239.67,
    "volume": 417124
  },
  {
    "date": "2025-09-17 10:20:00",
    "open": 239.84,
    "low": 239.7,
    "high": 240.02,
    "close": 239.76,
    "volume": 526900
  },
  {
    "date": "2025-09-17 10:15:00",
    "open": 239.52,
    "low": 239.49,
    "high": 239.89,
    "close": 239.84,
    "volume": 412939
  },
  {
    "date": "2025-09-17 10:10:00",
    "open": 239.52,
    "low": 239.47,
    "high": 239.73,
    "close": 239.53,
    "volume": 323404
  },
  {
    "date": "2025-09-17 10:05:00",
    "open": 239.66,
    "low": 239.42,
    "high": 239.93,
    "close": 239.5,
    "volume": 555106
  },
  {
    "date": "2025-09-17 10:00:00",
    "open": 239.2,
    "low": 239.06,
    "high": 239.71,
    "close": 239.69,
    "volume": 649674
  },
  {
    "date": "2025-09-17 09:55:00",
    "open": 239.54,
    "low": 238.94,
    "high": 239.74,
    "close": 239.23,
    "volume": 970406
  },
  {
    "date": "2025-09-17 09:50:00",
    "open": 239.68,
    "low": 239.43,
    "high": 239.85,
    "close": 239.52,
    "volume": 547400
  },
  {
    "date": "2025-09-17 09:45:00",
    "open": 239.82,
    "low": 239.33,
    "high": 239.84,
    "close": 239.67,
    "volume": 1931962
  },
  {
    "date": "2025-09-17 09:40:00",
    "open": 239.64,
    "low": 239.37,
    "high": 240.01,
    "close": 239.85,
    "volume": 659151
  },
  {
    "date": "2025-09-17 09:35:00",
    "open": 239.48,
    "low": 239.32,
    "high": 239.95,
    "close": 239.6,
    "volume": 688634
  },
  {
    "date": "2025-09-17 09:30:00",
    "open": 238.96,
    "low": 238.82,
    "high": 240.1,
    "close": 239.5,
    "volume": 2244704
  },
  {
    "date": "2025-09-16 15:55:00",
    "open": 238.73,
    "low": 238.08,
    "high": 238.76,
    "close": 238.14,
    "volume": 2359336
  },
  {
    "date": "2025-09-16 15:50:00",
    "open": 239.02,
    "low": 238.61,
    "high": 239.5,
    "close": 238.71,
    "volume": 1494816
  },
  {
    "date": "2025-09-16 15:45:00",
    "open": 238.95,
    "low": 238.88,
    "high": 239.16,
    "close": 239.02,
    "volume": 510563
  },
  {
    "date": "2025-09-16 15:40:00",
    "open": 239,
    "low": 238.9,
    "high": 239.17,
    "close": 238.94,
    "volume": 646807
  },
  {
    "date": "2025-09-16 15:35:00",
    "open": 239,
    "low": 238.92,
    "high": 239.08,
    "close": 239,
    "volume": 485603
  },
  {
    "date": "2025-09-16 15:30:00",
    "open": 238.94,
    "low": 238.81,
    "high": 239.07,
    "close": 238.99,
    "volume": 372496
  },
  {
    "date": "2025-09-16 15:25:00",
    "open": 239.14,
    "low": 238.85,
    "high": 239.15,
    "close": 238.94,
    "volume": 396689
  },
  {
    "date": "2025-09-16 15:20:00",
    "open": 239.33,
    "low": 238.92,
    "high": 239.33,
    "close": 239.15,
    "volume": 601415
  },
  {
    "date": "2025-09-16 15:15:00",
    "open": 239.54,
    "low": 239.3,
    "high": 239.6,
    "close": 239.33,
    "volume": 51900
  },
  {
    "date": "2025-09-16 15:10:00",
    "open": 239.25,
    "low": 239.19,
    "high": 239.58,
    "close": 239.54,
    "volume": 422975
  },
  {
    "date": "2025-09-16 15:05:00",
    "open": 239.21,
    "low": 239.18,
    "high": 239.38,
    "close": 239.25,
    "volume": 282719
  },
  {
    "date": "2025-09-16 15:00:00",
    "open": 239.06,
    "low": 239.05,
    "high": 239.26,
    "close": 239.19,
    "volume": 294282
  },
  {
    "date": "2025-09-16 14:55:00",
    "open": 238.96,
    "low": 238.91,
    "high": 239.11,
    "close": 239.07,
    "volume": 189172
  },
  {
    "date": "2025-09-16 14:50:00",
    "open": 239.01,
    "low": 238.93,
    "high": 239.07,
    "close": 238.97,
    "volume": 199419
  },
  {
    "date": "2025-09-16 14:45:00",
    "open": 239.02,
    "low": 239,
    "high": 239.09,
    "close": 239.02,
    "volume": 155721
  },
  {
    "date": "2025-09-16 14:40:00",
    "open": 238.95,
    "low": 238.94,
    "high": 239.08,
    "close": 239.01,
    "volume": 228300
  },
  {
    "date": "2025-09-16 14:35:00",
    "open": 238.94,
    "low": 238.81,
    "high": 238.96,
    "close": 238.95,
    "volume": 182250
  },
  {
    "date": "2025-09-16 14:30:00",
    "open": 238.94,
    "low": 238.82,
    "high": 239,
    "close": 238.94,
    "volume": 213106
  },
  {
    "date": "2025-09-16 14:25:00",
    "open": 238.86,
    "low": 238.82,
    "high": 239.02,
    "close": 238.93,
    "volume": 234245
  },
  {
    "date": "2025-09-16 14:20:00",
    "open": 238.82,
    "low": 238.66,
    "high": 238.86,
    "close": 238.85,
    "volume": 180503
  },
  {
    "date": "2025-09-16 14:15:00",
    "open": 238.79,
    "low": 238.71,
    "high": 238.88,
    "close": 238.8,
    "volume": 186876
  },
  {
    "date": "2025-09-16 14:10:00",
    "open": 238.85,
    "low": 238.71,
    "high": 238.96,
    "close": 238.78,
    "volume": 252568
  },
  {
    "date": "2025-09-16 14:05:00",
    "open": 238.49,
    "low": 238.46,
    "high": 238.87,
    "close": 238.84,
    "volume": 266807
  },
  {
    "date": "2025-09-16 14:00:00",
    "open": 238.28,
    "low": 238.21,
    "high": 238.48,
    "close": 238.48,
    "volume": 183071
  },
  {
    "date": "2025-09-16 13:55:00",
    "open": 238.47,
    "low": 238.24,
    "high": 238.47,
    "close": 238.3,
    "volume": 167001
  },
  {
    "date": "2025-09-16 13:50:00",
    "open": 238.36,
    "low": 238.13,
    "high": 238.47,
    "close": 238.47,
    "volume": 264186
  },
  {
    "date": "2025-09-16 13:45:00",
    "open": 238.39,
    "low": 238.22,
    "high": 238.41,
    "close": 238.36,
    "volume": 235094
  },
  {
    "date": "2025-09-16 13:40:00",
    "open": 238.61,
    "low": 238.31,
    "high": 238.65,
    "close": 238.39,
    "volume": 245014
  },
  {
    "date": "2025-09-16 13:35:00",
    "open": 238.74,
    "low": 238.58,
    "high": 238.78,
    "close": 238.62,
    "volume": 185589
  },
  {
    "date": "2025-09-16 13:30:00",
    "open": 239.11,
    "low": 238.67,
    "high": 239.11,
    "close": 238.72,
    "volume": 316150
  },
  {
    "date": "2025-09-16 13:25:00",
    "open": 239.21,
    "low": 239.05,
    "high": 239.23,
    "close": 239.09,
    "volume": 149973
  },
  {
    "date": "2025-09-16 13:20:00",
    "open": 239.2,
    "low": 239.11,
    "high": 239.36,
    "close": 239.21,
    "volume": 172122
  },
  {
    "date": "2025-09-16 13:15:00",
    "open": 239.14,
    "low": 239.07,
    "high": 239.39,
    "close": 239.21,
    "volume": 232856
  },
  {
    "date": "2025-09-16 13:10:00",
    "open": 239.02,
    "low": 238.99,
    "high": 239.22,
    "close": 239.15,
    "volume": 195878
  },
  {
    "date": "2025-09-16 13:05:00",
    "open": 239.19,
    "low": 238.98,
    "high": 239.22,
    "close": 239.02,
    "volume": 183656
  },
  {
    "date": "2025-09-16 13:00:00",
    "open": 239.15,
    "low": 239.15,
    "high": 239.4,
    "close": 239.18,
    "volume": 172402
  },
  {
    "date": "2025-09-16 12:55:00",
    "open": 239.09,
    "low": 239.01,
    "high": 239.3,
    "close": 239.15,
    "volume": 201128
  },
  {
    "date": "2025-09-16 12:50:00",
    "open": 239.15,
    "low": 239.05,
    "high": 239.34,
    "close": 239.09,
    "volume": 185410
  },
  {
    "date": "2025-09-16 12:45:00",
    "open": 239.25,
    "low": 239.05,
    "high": 239.3,
    "close": 239.14,
    "volume": 159466
  },
  {
    "date": "2025-09-16 12:40:00",
    "open": 239.12,
    "low": 239.07,
    "high": 239.35,
    "close": 239.26,
    "volume": 178739
  },
  {
    "date": "2025-09-16 12:35:00",
    "open": 239.22,
    "low": 238.9,
    "high": 239.22,
    "close": 239.12,
    "volume": 233995
  },
  {
    "date": "2025-09-16 12:30:00",
    "open": 239.21,
    "low": 239,
    "high": 239.26,
    "close": 239.2,
    "volume": 159055
  },
  {
    "date": "2025-09-16 12:25:00",
    "open": 239.35,
    "low": 239.04,
    "high": 239.36,
    "close": 239.2,
    "volume": 197790
  },
  {
    "date": "2025-09-16 12:20:00",
    "open": 239.18,
    "low": 239.11,
    "high": 239.4,
    "close": 239.36,
    "volume": 188553
  },
  {
    "date": "2025-09-16 12:15:00",
    "open": 239.05,
    "low": 238.99,
    "high": 239.31,
    "close": 239.2,
    "volume": 339729
  },
  {
    "date": "2025-09-16 12:10:00",
    "open": 238.85,
    "low": 238.76,
    "high": 239.09,
    "close": 239.07,
    "volume": 216390
  },
  {
    "date": "2025-09-16 12:05:00",
    "open": 238.85,
    "low": 238.76,
    "high": 238.92,
    "close": 238.89,
    "volume": 255837
  },
  {
    "date": "2025-09-16 12:00:00",
    "open": 238.67,
    "low": 238.63,
    "high": 238.87,
    "close": 238.84,
    "volume": 243572
  },
  {
    "date": "2025-09-16 11:55:00",
    "open": 238.77,
    "low": 238.67,
    "high": 238.96,
    "close": 238.68,
    "volume": 217521
  },
  {
    "date": "2025-09-16 11:50:00",
    "open": 238.96,
    "low": 238.63,
    "high": 239.02,
    "close": 238.77,
    "volume": 229672
  },
  {
    "date": "2025-09-16 11:45:00",
    "open": 238.68,
    "low": 238.68,
    "high": 239.03,
    "close": 238.92,
    "volume": 235791
  },
  {
    "date": "2025-09-16 11:40:00",
    "open": 238.83,
    "low": 238.47,
    "high": 238.92,
    "close": 238.67,
    "volume": 358806
  },
  {
    "date": "2025-09-16 11:35:00",
    "open": 239.28,
    "low": 238.69,
    "high": 239.29,
    "close": 238.82,
    "volume": 341929
  },
  {
    "date": "2025-09-16 11:30:00",
    "open": 239.59,
    "low": 239.1,
    "high": 239.59,
    "close": 239.23,
    "volume": 274967
  },
  {
    "date": "2025-09-16 11:25:00",
    "open": 239.64,
    "low": 239.45,
    "high": 239.73,
    "close": 239.61,
    "volume": 272744
  },
  {
    "date": "2025-09-16 11:20:00",
    "open": 239.57,
    "low": 239.5,
    "high": 239.85,
    "close": 239.62,
    "volume": 462328
  },
  {
    "date": "2025-09-16 11:15:00",
    "open": 239.36,
    "low": 239.01,
    "high": 239.62,
    "close": 239.56,
    "volume": 503021
  },
  {
    "date": "2025-09-16 11:10:00",
    "open": 238.77,
    "low": 238.7,
    "high": 239.36,
    "close": 239.36,
    "volume": 491709
  },
  {
    "date": "2025-09-16 11:05:00",
    "open": 238.41,
    "low": 238.25,
    "high": 238.87,
    "close": 238.77,
    "volume": 449969
  },
  {
    "date": "2025-09-16 11:00:00",
    "open": 238.99,
    "low": 238.27,
    "high": 239.05,
    "close": 238.41,
    "volume": 399553
  },
  {
    "date": "2025-09-16 10:55:00",
    "open": 238.98,
    "low": 238.82,
    "high": 239.24,
    "close": 238.98,
    "volume": 395495
  },
  {
    "date": "2025-09-16 10:50:00",
    "open": 238.95,
    "low": 238.73,
    "high": 239.17,
    "close": 238.99,
    "volume": 406309
  },
  {
    "date": "2025-09-16 10:45:00",
    "open": 239.23,
    "low": 238.87,
    "high": 239.27,
    "close": 238.93,
    "volume": 438336
  },
  {
    "date": "2025-09-16 10:40:00",
    "open": 239.46,
    "low": 239.18,
    "high": 239.6,
    "close": 239.25,
    "volume": 347522
  },
  {
    "date": "2025-09-16 10:35:00",
    "open": 239.38,
    "low": 239.18,
    "high": 239.68,
    "close": 239.46,
    "volume": 576201
  },
  {
    "date": "2025-09-16 10:30:00",
    "open": 239.52,
    "low": 239.17,
    "high": 239.73,
    "close": 239.37,
    "volume": 595091
  },
  {
    "date": "2025-09-16 10:25:00",
    "open": 240.22,
    "low": 239.48,
    "high": 240.34,
    "close": 239.53,
    "volume": 619861
  },
  {
    "date": "2025-09-16 10:20:00",
    "open": 240.87,
    "low": 239.83,
    "high": 240.97,
    "close": 240.19,
    "volume": 752134
  },
  {
    "date": "2025-09-16 10:15:00",
    "open": 240.68,
    "low": 240.66,
    "high": 241,
    "close": 240.85,
    "volume": 754673
  },
  {
    "date": "2025-09-16 10:10:00",
    "open": 240.61,
    "low": 240.16,
    "high": 240.79,
    "close": 240.67,
    "volume": 783416
  },
  {
    "date": "2025-09-16 10:05:00",
    "open": 240.99,
    "low": 240.34,
    "high": 241.06,
    "close": 240.61,
    "volume": 863968
  },
  {
    "date": "2025-09-16 10:00:00",
    "open": 240.4,
    "low": 240.38,
    "high": 241.21,
    "close": 241,
    "volume": 1456049
  },
  {
    "date": "2025-09-16 09:55:00",
    "open": 239.83,
    "low": 239.54,
    "high": 240.56,
    "close": 240.38,
    "volume": 1693218
  },
  {
    "date": "2025-09-16 09:50:00",
    "open": 239.1,
    "low": 238.81,
    "high": 239.85,
    "close": 239.83,
    "volume": 1312800
  },
  {
    "date": "2025-09-16 09:45:00",
    "open": 238.29,
    "low": 238.02,
    "high": 239.15,
    "close": 239.15,
    "volume": 1360736
  },
  {
    "date": "2025-09-16 09:40:00",
    "open": 237.03,
    "low": 237.02,
    "high": 238.33,
    "close": 238.29,
    "volume": 1062593
  },
  {
    "date": "2025-09-16 09:35:00",
    "open": 237,
    "low": 236.33,
    "high": 237.38,
    "close": 237.02,
    "volume": 806745
  },
  {
    "date": "2025-09-16 09:30:00",
    "open": 237.17,
    "low": 236.68,
    "high": 237.92,
    "close": 236.98,
    "volume": 2953084
  },
  {
    "date": "2025-09-15 15:55:00",
    "open": 235.95,
    "low": 235.86,
    "high": 236.79,
    "close": 236.76,
    "volume": 2432989
  },
  {
    "date": "2025-09-15 15:50:00",
    "open": 235.93,
    "low": 235.7,
    "high": 235.96,
    "close": 235.96,
    "volume": 657099
  },
  {
    "date": "2025-09-15 15:45:00",
    "open": 235.76,
    "low": 235.69,
    "high": 235.98,
    "close": 235.95,
    "volume": 415395
  },
  {
    "date": "2025-09-15 15:40:00",
    "open": 235.93,
    "low": 235.73,
    "high": 236.01,
    "close": 235.77,
    "volume": 427803
  },
  {
    "date": "2025-09-15 15:35:00",
    "open": 236.01,
    "low": 235.9,
    "high": 236.12,
    "close": 235.94,
    "volume": 342493
  },
  {
    "date": "2025-09-15 15:30:00",
    "open": 235.97,
    "low": 235.84,
    "high": 236.1,
    "close": 236.01,
    "volume": 456750
  },
  {
    "date": "2025-09-15 15:25:00",
    "open": 236,
    "low": 235.87,
    "high": 236.03,
    "close": 235.98,
    "volume": 342789
  },
  {
    "date": "2025-09-15 15:20:00",
    "open": 235.85,
    "low": 235.83,
    "high": 236.06,
    "close": 236,
    "volume": 263432
  },
  {
    "date": "2025-09-15 15:15:00",
    "open": 235.86,
    "low": 235.72,
    "high": 235.9,
    "close": 235.87,
    "volume": 273494
  },
  {
    "date": "2025-09-15 15:10:00",
    "open": 236.25,
    "low": 235.85,
    "high": 236.25,
    "close": 235.87,
    "volume": 287220
  },
  {
    "date": "2025-09-15 15:05:00",
    "open": 236.23,
    "low": 236.16,
    "high": 236.34,
    "close": 236.25,
    "volume": 307117
  },
  {
    "date": "2025-09-15 15:00:00",
    "open": 236.18,
    "low": 236.09,
    "high": 236.27,
    "close": 236.23,
    "volume": 335739
  },
  {
    "date": "2025-09-15 14:55:00",
    "open": 235.87,
    "low": 235.86,
    "high": 236.17,
    "close": 236.16,
    "volume": 400707
  },
  {
    "date": "2025-09-15 14:50:00",
    "open": 235.73,
    "low": 235.73,
    "high": 235.89,
    "close": 235.86,
    "volume": 259473
  },
  {
    "date": "2025-09-15 14:45:00",
    "open": 235.35,
    "low": 235.34,
    "high": 235.74,
    "close": 235.72,
    "volume": 283548
  },
  {
    "date": "2025-09-15 14:40:00",
    "open": 235.72,
    "low": 235.27,
    "high": 235.74,
    "close": 235.35,
    "volume": 358811
  },
  {
    "date": "2025-09-15 14:35:00",
    "open": 235.63,
    "low": 235.62,
    "high": 235.81,
    "close": 235.68,
    "volume": 210319
  },
  {
    "date": "2025-09-15 14:30:00",
    "open": 235.63,
    "low": 235.6,
    "high": 235.74,
    "close": 235.64,
    "volume": 247837
  },
  {
    "date": "2025-09-15 14:25:00",
    "open": 235.78,
    "low": 235.64,
    "high": 235.89,
    "close": 235.64,
    "volume": 206713
  },
  {
    "date": "2025-09-15 14:20:00",
    "open": 235.91,
    "low": 235.69,
    "high": 235.91,
    "close": 235.78,
    "volume": 221251
  },
  {
    "date": "2025-09-15 14:15:00",
    "open": 235.99,
    "low": 235.87,
    "high": 235.99,
    "close": 235.92,
    "volume": 204135
  },
  {
    "date": "2025-09-15 14:10:00",
    "open": 236.12,
    "low": 235.82,
    "high": 236.14,
    "close": 236,
    "volume": 246435
  },
  {
    "date": "2025-09-15 14:05:00",
    "open": 236.08,
    "low": 235.94,
    "high": 236.1,
    "close": 236.1,
    "volume": 215282
  },
  {
    "date": "2025-09-15 14:00:00",
    "open": 236.02,
    "low": 235.91,
    "high": 236.11,
    "close": 236.09,
    "volume": 232304
  },
  {
    "date": "2025-09-15 13:55:00",
    "open": 235.93,
    "low": 235.88,
    "high": 236.12,
    "close": 236.01,
    "volume": 239311
  },
  {
    "date": "2025-09-15 13:50:00",
    "open": 235.89,
    "low": 235.88,
    "high": 236.11,
    "close": 235.92,
    "volume": 330874
  },
  {
    "date": "2025-09-15 13:45:00",
    "open": 235.71,
    "low": 235.63,
    "high": 235.92,
    "close": 235.88,
    "volume": 285259
  },
  {
    "date": "2025-09-15 13:40:00",
    "open": 235.72,
    "low": 235.71,
    "high": 235.86,
    "close": 235.72,
    "volume": 241044
  },
  {
    "date": "2025-09-15 13:35:00",
    "open": 235.68,
    "low": 235.59,
    "high": 235.75,
    "close": 235.73,
    "volume": 282143
  },
  {
    "date": "2025-09-15 13:30:00",
    "open": 235.75,
    "low": 235.62,
    "high": 235.8,
    "close": 235.69,
    "volume": 249003
  },
  {
    "date": "2025-09-15 13:25:00",
    "open": 235.88,
    "low": 235.76,
    "high": 235.97,
    "close": 235.76,
    "volume": 199800
  },
  {
    "date": "2025-09-15 13:20:00",
    "open": 235.84,
    "low": 235.7,
    "high": 235.92,
    "close": 235.86,
    "volume": 27037
  },
  {
    "date": "2025-09-15 13:15:00",
    "open": 236.13,
    "low": 235.84,
    "high": 236.22,
    "close": 235.84,
    "volume": 297118
  },
  {
    "date": "2025-09-15 13:10:00",
    "open": 236.15,
    "low": 236.03,
    "high": 236.23,
    "close": 236.11,
    "volume": 374610
  },
  {
    "date": "2025-09-15 13:05:00",
    "open": 236.28,
    "low": 236.15,
    "high": 236.37,
    "close": 236.15,
    "volume": 180722
  },
  {
    "date": "2025-09-15 13:00:00",
    "open": 236.23,
    "low": 236.13,
    "high": 236.35,
    "close": 236.28,
    "volume": 215894
  },
  {
    "date": "2025-09-15 12:55:00",
    "open": 236.14,
    "low": 236.13,
    "high": 236.32,
    "close": 236.23,
    "volume": 218391
  },
  {
    "date": "2025-09-15 12:50:00",
    "open": 236.39,
    "low": 236.08,
    "high": 236.45,
    "close": 236.14,
    "volume": 219702
  },
  {
    "date": "2025-09-15 12:45:00",
    "open": 236.42,
    "low": 236.3,
    "high": 236.44,
    "close": 236.4,
    "volume": 199925
  },
  {
    "date": "2025-09-15 12:40:00",
    "open": 236.52,
    "low": 236.35,
    "high": 236.52,
    "close": 236.43,
    "volume": 238096
  },
  {
    "date": "2025-09-15 12:35:00",
    "open": 236.43,
    "low": 236.21,
    "high": 236.5,
    "close": 236.5,
    "volume": 308279
  },
  {
    "date": "2025-09-15 12:30:00",
    "open": 236.57,
    "low": 236.43,
    "high": 236.59,
    "close": 236.44,
    "volume": 197543
  },
  {
    "date": "2025-09-15 12:25:00",
    "open": 236.58,
    "low": 236.48,
    "high": 236.62,
    "close": 236.57,
    "volume": 208458
  },
  {
    "date": "2025-09-15 12:20:00",
    "open": 236.62,
    "low": 236.56,
    "high": 236.67,
    "close": 236.57,
    "volume": 227660
  },
  {
    "date": "2025-09-15 12:15:00",
    "open": 236.79,
    "low": 236.63,
    "high": 236.83,
    "close": 236.64,
    "volume": 248326
  },
  {
    "date": "2025-09-15 12:10:00",
    "open": 236.79,
    "low": 236.71,
    "high": 236.89,
    "close": 236.78,
    "volume": 205604
  },
  {
    "date": "2025-09-15 12:05:00",
    "open": 236.93,
    "low": 236.77,
    "high": 237,
    "close": 236.78,
    "volume": 232302
  },
  {
    "date": "2025-09-15 12:00:00",
    "open": 237.04,
    "low": 236.7,
    "high": 237.05,
    "close": 236.94,
    "volume": 331314
  },
  {
    "date": "2025-09-15 11:55:00",
    "open": 237.09,
    "low": 237.02,
    "high": 237.27,
    "close": 237.09,
    "volume": 256535
  },
  {
    "date": "2025-09-15 11:50:00",
    "open": 237.08,
    "low": 236.96,
    "high": 237.19,
    "close": 237.08,
    "volume": 297647
  },
  {
    "date": "2025-09-15 11:45:00",
    "open": 236.9,
    "low": 236.76,
    "high": 237.14,
    "close": 237.11,
    "volume": 358713
  },
  {
    "date": "2025-09-15 11:40:00",
    "open": 236.99,
    "low": 236.91,
    "high": 237.31,
    "close": 236.91,
    "volume": 471246
  },
  {
    "date": "2025-09-15 11:35:00",
    "open": 236.93,
    "low": 236.74,
    "high": 237.05,
    "close": 236.98,
    "volume": 323421
  },
  {
    "date": "2025-09-15 11:30:00",
    "open": 236.83,
    "low": 236.78,
    "high": 237.12,
    "close": 236.93,
    "volume": 525277
  },
  {
    "date": "2025-09-15 11:25:00",
    "open": 236.55,
    "low": 236.55,
    "high": 236.89,
    "close": 236.82,
    "volume": 316831
  },
  {
    "date": "2025-09-15 11:20:00",
    "open": 236.57,
    "low": 236.52,
    "high": 236.85,
    "close": 236.54,
    "volume": 321794
  },
  {
    "date": "2025-09-15 11:15:00",
    "open": 236.64,
    "low": 236.53,
    "high": 236.69,
    "close": 236.57,
    "volume": 283101
  },
  {
    "date": "2025-09-15 11:10:00",
    "open": 236.62,
    "low": 236.6,
    "high": 236.8,
    "close": 236.65,
    "volume": 306249
  },
  {
    "date": "2025-09-15 11:05:00",
    "open": 236.75,
    "low": 236.42,
    "high": 236.75,
    "close": 236.63,
    "volume": 345506
  },
  {
    "date": "2025-09-15 11:00:00",
    "open": 236.96,
    "low": 236.57,
    "high": 236.99,
    "close": 236.76,
    "volume": 487155
  },
  {
    "date": "2025-09-15 10:55:00",
    "open": 236.7,
    "low": 236.68,
    "high": 237.02,
    "close": 236.95,
    "volume": 534615
  },
  {
    "date": "2025-09-15 10:50:00",
    "open": 236.27,
    "low": 236.25,
    "high": 236.77,
    "close": 236.7,
    "volume": 583955
  },
  {
    "date": "2025-09-15 10:45:00",
    "open": 236.45,
    "low": 236.22,
    "high": 236.45,
    "close": 236.28,
    "volume": 359924
  },
  {
    "date": "2025-09-15 10:40:00",
    "open": 235.99,
    "low": 235.94,
    "high": 236.62,
    "close": 236.47,
    "volume": 616177
  },
  {
    "date": "2025-09-15 10:35:00",
    "open": 236.04,
    "low": 235.66,
    "high": 236.05,
    "close": 235.98,
    "volume": 427688
  },
  {
    "date": "2025-09-15 10:30:00",
    "open": 236.22,
    "low": 235.88,
    "high": 236.28,
    "close": 236.02,
    "volume": 410171
  },
  {
    "date": "2025-09-15 10:25:00",
    "open": 235.75,
    "low": 235.73,
    "high": 236.29,
    "close": 236.21,
    "volume": 759488
  },
  {
    "date": "2025-09-15 10:20:00",
    "open": 235.34,
    "low": 235.26,
    "high": 235.8,
    "close": 235.72,
    "volume": 572669
  },
  {
    "date": "2025-09-15 10:15:00",
    "open": 235.37,
    "low": 235.3,
    "high": 235.47,
    "close": 235.35,
    "volume": 381741
  },
  {
    "date": "2025-09-15 10:10:00",
    "open": 235.41,
    "low": 235.15,
    "high": 235.48,
    "close": 235.36,
    "volume": 443983
  },
  {
    "date": "2025-09-15 10:05:00",
    "open": 235.41,
    "low": 235.06,
    "high": 235.61,
    "close": 235.4,
    "volume": 601768
  },
  {
    "date": "2025-09-15 10:00:00",
    "open": 235.36,
    "low": 235.31,
    "high": 235.65,
    "close": 235.39,
    "volume": 640927
  },
  {
    "date": "2025-09-15 09:55:00",
    "open": 235.52,
    "low": 235.03,
    "high": 235.6,
    "close": 235.35,
    "volume": 868529
  },
  {
    "date": "2025-09-15 09:50:00",
    "open": 235.2,
    "low": 235.16,
    "high": 235.79,
    "close": 235.54,
    "volume": 826128
  },
  {
    "date": "2025-09-15 09:45:00",
    "open": 235.73,
    "low": 235.08,
    "high": 235.93,
    "close": 235.19,
    "volume": 3137100
  },
  {
    "date": "2025-09-15 09:40:00",
    "open": 237.37,
    "low": 235.68,
    "high": 237.37,
    "close": 235.68,
    "volume": 1460331
  },
  {
    "date": "2025-09-15 09:35:00",
    "open": 237.24,
    "low": 237.2,
    "high": 238.19,
    "close": 237.35,
    "volume": 1593815
  },
  {
    "date": "2025-09-15 09:30:00",
    "open": 237,
    "low": 236.69,
    "high": 238.02,
    "close": 237.23,
    "volume": 4769892
  },
  {
    "date": "2025-09-12 15:55:00",
    "open": 234.08,
    "low": 233.99,
    "high": 234.29,
    "close": 234.05,
    "volume": 2977514
  },
  {
    "date": "2025-09-12 15:50:00",
    "open": 233.86,
    "low": 233.72,
    "high": 234.2,
    "close": 234.09,
    "volume": 917106
  },
  {
    "date": "2025-09-12 15:45:00",
    "open": 233.76,
    "low": 233.74,
    "high": 233.88,
    "close": 233.84,
    "volume": 449032
  },
  {
    "date": "2025-09-12 15:40:00",
    "open": 233.77,
    "low": 233.51,
    "high": 233.95,
    "close": 233.76,
    "volume": 576911
  },
  {
    "date": "2025-09-12 15:35:00",
    "open": 233.85,
    "low": 233.7,
    "high": 233.88,
    "close": 233.77,
    "volume": 397547
  },
  {
    "date": "2025-09-12 15:30:00",
    "open": 233.78,
    "low": 233.61,
    "high": 233.89,
    "close": 233.86,
    "volume": 556623
  },
  {
    "date": "2025-09-12 15:25:00",
    "open": 234.01,
    "low": 233.73,
    "high": 234.07,
    "close": 233.76,
    "volume": 320840
  },
  {
    "date": "2025-09-12 15:20:00",
    "open": 234.18,
    "low": 234,
    "high": 234.21,
    "close": 234.03,
    "volume": 295285
  },
  {
    "date": "2025-09-12 15:15:00",
    "open": 234.39,
    "low": 233.99,
    "high": 234.41,
    "close": 234.19,
    "volume": 401011
  },
  {
    "date": "2025-09-12 15:10:00",
    "open": 234.13,
    "low": 234.07,
    "high": 234.44,
    "close": 234.37,
    "volume": 494088
  },
  {
    "date": "2025-09-12 15:05:00",
    "open": 234.37,
    "low": 234.12,
    "high": 234.4,
    "close": 234.15,
    "volume": 339578
  },
  {
    "date": "2025-09-12 15:00:00",
    "open": 234.01,
    "low": 234.01,
    "high": 234.42,
    "close": 234.38,
    "volume": 454856
  },
  {
    "date": "2025-09-12 14:55:00",
    "open": 233.76,
    "low": 233.67,
    "high": 234.05,
    "close": 233.99,
    "volume": 489594
  },
  {
    "date": "2025-09-12 14:50:00",
    "open": 233.44,
    "low": 233.44,
    "high": 233.77,
    "close": 233.74,
    "volume": 1141845
  },
  {
    "date": "2025-09-12 14:45:00",
    "open": 234.17,
    "low": 233.02,
    "high": 234.22,
    "close": 233.35,
    "volume": 1237301
  },
  {
    "date": "2025-09-12 14:40:00",
    "open": 233.94,
    "low": 233.9,
    "high": 234.19,
    "close": 234.16,
    "volume": 450553
  },
  {
    "date": "2025-09-12 14:35:00",
    "open": 233.73,
    "low": 233.63,
    "high": 233.95,
    "close": 233.95,
    "volume": 996845
  },
  {
    "date": "2025-09-12 14:30:00",
    "open": 233.55,
    "low": 233.45,
    "high": 233.76,
    "close": 233.73,
    "volume": 349055
  },
  {
    "date": "2025-09-12 14:25:00",
    "open": 233.47,
    "low": 233.44,
    "high": 233.55,
    "close": 233.55,
    "volume": 197501
  },
  {
    "date": "2025-09-12 14:20:00",
    "open": 233.64,
    "low": 233.37,
    "high": 233.68,
    "close": 233.45,
    "volume": 256719
  },
  {
    "date": "2025-09-12 14:15:00",
    "open": 233.53,
    "low": 233.46,
    "high": 233.71,
    "close": 233.65,
    "volume": 363200
  },
  {
    "date": "2025-09-12 14:10:00",
    "open": 233.38,
    "low": 233.32,
    "high": 233.56,
    "close": 233.53,
    "volume": 268752
  },
  {
    "date": "2025-09-12 14:05:00",
    "open": 233.4,
    "low": 233.34,
    "high": 233.51,
    "close": 233.37,
    "volume": 245882
  },
  {
    "date": "2025-09-12 14:00:00",
    "open": 233.68,
    "low": 233.24,
    "high": 233.71,
    "close": 233.42,
    "volume": 273034
  },
  {
    "date": "2025-09-12 13:55:00",
    "open": 233.63,
    "low": 233.63,
    "high": 233.79,
    "close": 233.67,
    "volume": 241604
  },
  {
    "date": "2025-09-12 13:50:00",
    "open": 233.5,
    "low": 233.5,
    "high": 233.86,
    "close": 233.62,
    "volume": 345123
  },
  {
    "date": "2025-09-12 13:45:00",
    "open": 233.43,
    "low": 233.37,
    "high": 233.52,
    "close": 233.5,
    "volume": 228264
  },
  {
    "date": "2025-09-12 13:40:00",
    "open": 233.54,
    "low": 233.42,
    "high": 233.59,
    "close": 233.42,
    "volume": 255990
  },
  {
    "date": "2025-09-12 13:35:00",
    "open": 233.32,
    "low": 233.31,
    "high": 233.61,
    "close": 233.54,
    "volume": 325548
  },
  {
    "date": "2025-09-12 13:30:00",
    "open": 233.23,
    "low": 233.22,
    "high": 233.37,
    "close": 233.33,
    "volume": 202262
  },
  {
    "date": "2025-09-12 13:25:00",
    "open": 233.37,
    "low": 233.22,
    "high": 233.45,
    "close": 233.23,
    "volume": 250361
  },
  {
    "date": "2025-09-12 13:20:00",
    "open": 233.19,
    "low": 233.07,
    "high": 233.45,
    "close": 233.39,
    "volume": 241901
  },
  {
    "date": "2025-09-12 13:15:00",
    "open": 233.13,
    "low": 233.05,
    "high": 233.43,
    "close": 233.21,
    "volume": 380409
  },
  {
    "date": "2025-09-12 13:10:00",
    "open": 233.16,
    "low": 233.14,
    "high": 233.28,
    "close": 233.14,
    "volume": 220708
  },
  {
    "date": "2025-09-12 13:05:00",
    "open": 233.09,
    "low": 233.06,
    "high": 233.22,
    "close": 233.17,
    "volume": 200900
  },
  {
    "date": "2025-09-12 13:00:00",
    "open": 233.14,
    "low": 233,
    "high": 233.21,
    "close": 233.06,
    "volume": 797302
  },
  {
    "date": "2025-09-12 12:55:00",
    "open": 233.42,
    "low": 233.13,
    "high": 233.42,
    "close": 233.14,
    "volume": 1424620
  },
  {
    "date": "2025-09-12 12:50:00",
    "open": 233.76,
    "low": 233.43,
    "high": 233.77,
    "close": 233.43,
    "volume": 388776
  },
  {
    "date": "2025-09-12 12:45:00",
    "open": 233.73,
    "low": 233.66,
    "high": 233.83,
    "close": 233.75,
    "volume": 197753
  },
  {
    "date": "2025-09-12 12:40:00",
    "open": 233.86,
    "low": 233.71,
    "high": 233.87,
    "close": 233.74,
    "volume": 274774
  },
  {
    "date": "2025-09-12 12:35:00",
    "open": 233.82,
    "low": 233.76,
    "high": 233.96,
    "close": 233.86,
    "volume": 240346
  },
  {
    "date": "2025-09-12 12:30:00",
    "open": 233.95,
    "low": 233.61,
    "high": 233.95,
    "close": 233.83,
    "volume": 351554
  },
  {
    "date": "2025-09-12 12:25:00",
    "open": 234.02,
    "low": 233.94,
    "high": 234.06,
    "close": 233.96,
    "volume": 254163
  },
  {
    "date": "2025-09-12 12:20:00",
    "open": 234.06,
    "low": 233.95,
    "high": 234.09,
    "close": 234.02,
    "volume": 247170
  },
  {
    "date": "2025-09-12 12:15:00",
    "open": 234.09,
    "low": 234,
    "high": 234.26,
    "close": 234.03,
    "volume": 306995
  },
  {
    "date": "2025-09-12 12:10:00",
    "open": 234.05,
    "low": 233.97,
    "high": 234.18,
    "close": 234.08,
    "volume": 361141
  },
  {
    "date": "2025-09-12 12:05:00",
    "open": 233.92,
    "low": 233.77,
    "high": 234.13,
    "close": 234.07,
    "volume": 446399
  },
  {
    "date": "2025-09-12 12:00:00",
    "open": 234,
    "low": 233.83,
    "high": 234.1,
    "close": 233.91,
    "volume": 433938
  },
  {
    "date": "2025-09-12 11:55:00",
    "open": 234.39,
    "low": 233.99,
    "high": 234.39,
    "close": 233.99,
    "volume": 332019
  },
  {
    "date": "2025-09-12 11:50:00",
    "open": 234.28,
    "low": 234.2,
    "high": 234.48,
    "close": 234.39,
    "volume": 402735
  },
  {
    "date": "2025-09-12 11:45:00",
    "open": 233.86,
    "low": 233.85,
    "high": 234.32,
    "close": 234.28,
    "volume": 798392
  },
  {
    "date": "2025-09-12 11:40:00",
    "open": 233.73,
    "low": 233.67,
    "high": 234.11,
    "close": 233.87,
    "volume": 311167
  },
  {
    "date": "2025-09-12 11:35:00",
    "open": 234.09,
    "low": 233.62,
    "high": 234.12,
    "close": 233.75,
    "volume": 382564
  },
  {
    "date": "2025-09-12 11:30:00",
    "open": 234.06,
    "low": 233.93,
    "high": 234.26,
    "close": 234.13,
    "volume": 434098
  },
  {
    "date": "2025-09-12 11:25:00",
    "open": 233.96,
    "low": 233.87,
    "high": 234.22,
    "close": 234.07,
    "volume": 514595
  },
  {
    "date": "2025-09-12 11:20:00",
    "open": 233.55,
    "low": 233.38,
    "high": 233.96,
    "close": 233.96,
    "volume": 6577687
  },
  {
    "date": "2025-09-12 11:15:00",
    "open": 233.9,
    "low": 233.53,
    "high": 234.09,
    "close": 233.55,
    "volume": 452684
  },
  {
    "date": "2025-09-12 11:10:00",
    "open": 233.75,
    "low": 233.58,
    "high": 233.96,
    "close": 233.89,
    "volume": 282780
  },
  {
    "date": "2025-09-12 11:05:00",
    "open": 233.95,
    "low": 233.56,
    "high": 234.22,
    "close": 233.76,
    "volume": 346062
  },
  {
    "date": "2025-09-12 11:00:00",
    "open": 233.95,
    "low": 233.88,
    "high": 234.29,
    "close": 233.95,
    "volume": 689999
  },
  {
    "date": "2025-09-12 10:55:00",
    "open": 233.71,
    "low": 233.68,
    "high": 234.13,
    "close": 233.93,
    "volume": 563205
  },
  {
    "date": "2025-09-12 10:50:00",
    "open": 233.84,
    "low": 233.46,
    "high": 233.86,
    "close": 233.75,
    "volume": 675375
  },
  {
    "date": "2025-09-12 10:45:00",
    "open": 233.95,
    "low": 233.77,
    "high": 234.17,
    "close": 233.83,
    "volume": 542061
  },
  {
    "date": "2025-09-12 10:40:00",
    "open": 233.98,
    "low": 233.84,
    "high": 234.51,
    "close": 233.9,
    "volume": 879842
  },
  {
    "date": "2025-09-12 10:35:00",
    "open": 234.22,
    "low": 233.68,
    "high": 234.34,
    "close": 233.97,
    "volume": 876166
  },
  {
    "date": "2025-09-12 10:30:00",
    "open": 234.07,
    "low": 233.52,
    "high": 234.48,
    "close": 234.24,
    "volume": 1159636
  },
  {
    "date": "2025-09-12 10:25:00",
    "open": 233.66,
    "low": 233.54,
    "high": 234.07,
    "close": 234.06,
    "volume": 1273686
  },
  {
    "date": "2025-09-12 10:20:00",
    "open": 232.77,
    "low": 232.53,
    "high": 233.79,
    "close": 233.65,
    "volume": 1208901
  },
  {
    "date": "2025-09-12 10:15:00",
    "open": 232.44,
    "low": 232.32,
    "high": 232.83,
    "close": 232.73,
    "volume": 851509
  },
  {
    "date": "2025-09-12 10:10:00",
    "open": 232.69,
    "low": 232.15,
    "high": 232.79,
    "close": 232.36,
    "volume": 1099100
  },
  {
    "date": "2025-09-12 10:05:00",
    "open": 231.52,
    "low": 231.49,
    "high": 232.75,
    "close": 232.71,
    "volume": 1626882
  },
  {
    "date": "2025-09-12 10:00:00",
    "open": 231.19,
    "low": 230.92,
    "high": 231.61,
    "close": 231.51,
    "volume": 900158
  },
  {
    "date": "2025-09-12 09:55:00",
    "open": 230.39,
    "low": 230.24,
    "high": 231.23,
    "close": 231.23,
    "volume": 668665
  },
  {
    "date": "2025-09-12 09:50:00",
    "open": 229.78,
    "low": 229.75,
    "high": 230.43,
    "close": 230.39,
    "volume": 672728
  },
  {
    "date": "2025-09-12 09:45:00",
    "open": 229.16,
    "low": 229.1,
    "high": 230.04,
    "close": 229.79,
    "volume": 1979639
  },
  {
    "date": "2025-09-12 09:40:00",
    "open": 229.51,
    "low": 229.04,
    "high": 229.63,
    "close": 229.19,
    "volume": 688382
  },
  {
    "date": "2025-09-12 09:35:00",
    "open": 230.48,
    "low": 229.2,
    "high": 230.61,
    "close": 229.48,
    "volume": 966002
  },
  {
    "date": "2025-09-12 09:30:00",
    "open": 229.3,
    "low": 229.17,
    "high": 230.93,
    "close": 230.5,
    "volume": 4178856
  },
  {
    "date": "2025-09-11 15:55:00",
    "open": 230.39,
    "low": 229.94,
    "high": 230.45,
    "close": 229.99,
    "volume": 2243509
  },
  {
    "date": "2025-09-11 15:50:00",
    "open": 230.35,
    "low": 229.78,
    "high": 230.44,
    "close": 230.4,
    "volume": 121715
  },
  {
    "date": "2025-09-11 15:45:00",
    "open": 230.05,
    "low": 230.04,
    "high": 230.37,
    "close": 230.35,
    "volume": 636513
  },
  {
    "date": "2025-09-11 15:40:00",
    "open": 230.09,
    "low": 230.02,
    "high": 230.14,
    "close": 230.04,
    "volume": 496894
  },
  {
    "date": "2025-09-11 15:35:00",
    "open": 230.08,
    "low": 230.03,
    "high": 230.18,
    "close": 230.09,
    "volume": 472365
  },
  {
    "date": "2025-09-11 15:30:00",
    "open": 229.84,
    "low": 229.84,
    "high": 230.09,
    "close": 230.08,
    "volume": 579373
  },
  {
    "date": "2025-09-11 15:25:00",
    "open": 229.8,
    "low": 229.8,
    "high": 229.91,
    "close": 229.85,
    "volume": 300810
  },
  {
    "date": "2025-09-11 15:20:00",
    "open": 229.6,
    "low": 229.56,
    "high": 229.83,
    "close": 229.8,
    "volume": 316494
  },
  {
    "date": "2025-09-11 15:15:00",
    "open": 229.66,
    "low": 229.54,
    "high": 229.71,
    "close": 229.6,
    "volume": 304807
  },
  {
    "date": "2025-09-11 15:10:00",
    "open": 229.72,
    "low": 229.42,
    "high": 229.75,
    "close": 229.65,
    "volume": 492713
  },
  {
    "date": "2025-09-11 15:05:00",
    "open": 230.03,
    "low": 229.71,
    "high": 230.05,
    "close": 229.73,
    "volume": 302632
  },
  {
    "date": "2025-09-11 15:00:00",
    "open": 229.87,
    "low": 229.86,
    "high": 230.06,
    "close": 230.03,
    "volume": 332053
  },
  {
    "date": "2025-09-11 14:55:00",
    "open": 229.98,
    "low": 229.76,
    "high": 229.98,
    "close": 229.86,
    "volume": 347361
  },
  {
    "date": "2025-09-11 14:50:00",
    "open": 230.1,
    "low": 229.91,
    "high": 230.15,
    "close": 229.99,
    "volume": 287914
  },
  {
    "date": "2025-09-11 14:45:00",
    "open": 229.96,
    "low": 229.87,
    "high": 230.14,
    "close": 230.08,
    "volume": 366839
  },
  {
    "date": "2025-09-11 14:40:00",
    "open": 229.91,
    "low": 229.85,
    "high": 229.98,
    "close": 229.97,
    "volume": 209275
  },
  {
    "date": "2025-09-11 14:35:00",
    "open": 229.73,
    "low": 229.71,
    "high": 229.92,
    "close": 229.92,
    "volume": 374746
  },
  {
    "date": "2025-09-11 14:30:00",
    "open": 229.46,
    "low": 229.45,
    "high": 229.75,
    "close": 229.72,
    "volume": 249162
  },
  {
    "date": "2025-09-11 14:25:00",
    "open": 229.34,
    "low": 229.32,
    "high": 229.51,
    "close": 229.46,
    "volume": 308437
  },
  {
    "date": "2025-09-11 14:20:00",
    "open": 229.51,
    "low": 229.19,
    "high": 229.56,
    "close": 229.35,
    "volume": 413021
  },
  {
    "date": "2025-09-11 14:15:00",
    "open": 229.56,
    "low": 229.42,
    "high": 229.61,
    "close": 229.5,
    "volume": 483086
  },
  {
    "date": "2025-09-11 14:10:00",
    "open": 229.86,
    "low": 229.52,
    "high": 229.86,
    "close": 229.56,
    "volume": 328911
  },
  {
    "date": "2025-09-11 14:05:00",
    "open": 229.74,
    "low": 229.72,
    "high": 230.04,
    "close": 229.87,
    "volume": 464576
  },
  {
    "date": "2025-09-11 14:00:00",
    "open": 230.07,
    "low": 229.7,
    "high": 230.08,
    "close": 229.71,
    "volume": 483766
  },
  {
    "date": "2025-09-11 13:55:00",
    "open": 229.94,
    "low": 229.89,
    "high": 230.16,
    "close": 230.06,
    "volume": 627109
  },
  {
    "date": "2025-09-11 13:50:00",
    "open": 229.8,
    "low": 229.64,
    "high": 229.95,
    "close": 229.95,
    "volume": 340774
  },
  {
    "date": "2025-09-11 13:45:00",
    "open": 229.78,
    "low": 229.6,
    "high": 229.83,
    "close": 229.8,
    "volume": 313191
  },
  {
    "date": "2025-09-11 13:40:00",
    "open": 229.54,
    "low": 229.54,
    "high": 229.81,
    "close": 229.78,
    "volume": 311387
  },
  {
    "date": "2025-09-11 13:35:00",
    "open": 229.53,
    "low": 229.48,
    "high": 229.64,
    "close": 229.51,
    "volume": 242617
  },
  {
    "date": "2025-09-11 13:30:00",
    "open": 229.47,
    "low": 229.39,
    "high": 229.68,
    "close": 229.53,
    "volume": 346630
  },
  {
    "date": "2025-09-11 13:25:00",
    "open": 229.31,
    "low": 229.31,
    "high": 229.54,
    "close": 229.48,
    "volume": 415239
  },
  {
    "date": "2025-09-11 13:20:00",
    "open": 229.19,
    "low": 229.05,
    "high": 229.38,
    "close": 229.33,
    "volume": 315301
  },
  {
    "date": "2025-09-11 13:15:00",
    "open": 229.17,
    "low": 229.15,
    "high": 229.37,
    "close": 229.22,
    "volume": 301791
  },
  {
    "date": "2025-09-11 13:10:00",
    "open": 229.1,
    "low": 229.06,
    "high": 229.32,
    "close": 229.18,
    "volume": 412891
  },
  {
    "date": "2025-09-11 13:05:00",
    "open": 228.97,
    "low": 228.8,
    "high": 229.1,
    "close": 229.09,
    "volume": 365217
  },
  {
    "date": "2025-09-11 13:00:00",
    "open": 228.76,
    "low": 228.65,
    "high": 228.96,
    "close": 228.95,
    "volume": 411143
  },
  {
    "date": "2025-09-11 12:55:00",
    "open": 228.65,
    "low": 228.58,
    "high": 228.78,
    "close": 228.75,
    "volume": 219327
  },
  {
    "date": "2025-09-11 12:50:00",
    "open": 228.68,
    "low": 228.31,
    "high": 228.68,
    "close": 228.65,
    "volume": 211191
  },
  {
    "date": "2025-09-11 12:45:00",
    "open": 228.6,
    "low": 228.55,
    "high": 228.72,
    "close": 228.69,
    "volume": 247682
  },
  {
    "date": "2025-09-11 12:40:00",
    "open": 228.39,
    "low": 228.39,
    "high": 228.66,
    "close": 228.59,
    "volume": 263638
  },
  {
    "date": "2025-09-11 12:35:00",
    "open": 228.61,
    "low": 228.38,
    "high": 228.61,
    "close": 228.38,
    "volume": 161060
  },
  {
    "date": "2025-09-11 12:30:00",
    "open": 228.62,
    "low": 228.54,
    "high": 228.67,
    "close": 228.62,
    "volume": 191919
  },
  {
    "date": "2025-09-11 12:25:00",
    "open": 228.64,
    "low": 228.6,
    "high": 228.76,
    "close": 228.61,
    "volume": 209827
  },
  {
    "date": "2025-09-11 12:20:00",
    "open": 228.6,
    "low": 228.54,
    "high": 228.72,
    "close": 228.65,
    "volume": 204466
  },
  {
    "date": "2025-09-11 12:15:00",
    "open": 228.29,
    "low": 228.24,
    "high": 228.6,
    "close": 228.6,
    "volume": 277365
  },
  {
    "date": "2025-09-11 12:10:00",
    "open": 228.2,
    "low": 228.15,
    "high": 228.29,
    "close": 228.28,
    "volume": 255892
  },
  {
    "date": "2025-09-11 12:05:00",
    "open": 228.38,
    "low": 228.04,
    "high": 228.39,
    "close": 228.21,
    "volume": 292757
  },
  {
    "date": "2025-09-11 12:00:00",
    "open": 228.4,
    "low": 228.26,
    "high": 228.51,
    "close": 228.38,
    "volume": 275768
  },
  {
    "date": "2025-09-11 11:55:00",
    "open": 228.41,
    "low": 228.19,
    "high": 228.49,
    "close": 228.41,
    "volume": 355715
  },
  {
    "date": "2025-09-11 11:50:00",
    "open": 228.73,
    "low": 228.37,
    "high": 228.77,
    "close": 228.39,
    "volume": 355488
  },
  {
    "date": "2025-09-11 11:45:00",
    "open": 228.68,
    "low": 228.57,
    "high": 228.76,
    "close": 228.74,
    "volume": 237155
  },
  {
    "date": "2025-09-11 11:40:00",
    "open": 228.44,
    "low": 228.37,
    "high": 228.69,
    "close": 228.67,
    "volume": 342604
  },
  {
    "date": "2025-09-11 11:35:00",
    "open": 228.55,
    "low": 228.33,
    "high": 228.55,
    "close": 228.43,
    "volume": 356846
  },
  {
    "date": "2025-09-11 11:30:00",
    "open": 228.74,
    "low": 228.52,
    "high": 228.75,
    "close": 228.55,
    "volume": 6822019
  },
  {
    "date": "2025-09-11 11:25:00",
    "open": 228.71,
    "low": 228.46,
    "high": 228.77,
    "close": 228.77,
    "volume": 269039
  },
  {
    "date": "2025-09-11 11:20:00",
    "open": 228.71,
    "low": 228.46,
    "high": 228.81,
    "close": 228.72,
    "volume": 351338
  },
  {
    "date": "2025-09-11 11:15:00",
    "open": 228.69,
    "low": 228.41,
    "high": 228.82,
    "close": 228.7,
    "volume": 403756
  },
  {
    "date": "2025-09-11 11:10:00",
    "open": 228.71,
    "low": 228.61,
    "high": 228.88,
    "close": 228.71,
    "volume": 364659
  },
  {
    "date": "2025-09-11 11:05:00",
    "open": 229.1,
    "low": 228.7,
    "high": 229.21,
    "close": 228.71,
    "volume": 380667
  },
  {
    "date": "2025-09-11 11:00:00",
    "open": 229.01,
    "low": 228.94,
    "high": 229.32,
    "close": 229.11,
    "volume": 390425
  },
  {
    "date": "2025-09-11 10:55:00",
    "open": 228.75,
    "low": 228.6,
    "high": 229.08,
    "close": 229.03,
    "volume": 415855
  },
  {
    "date": "2025-09-11 10:50:00",
    "open": 228.47,
    "low": 228.45,
    "high": 228.82,
    "close": 228.76,
    "volume": 2032337
  },
  {
    "date": "2025-09-11 10:45:00",
    "open": 228.65,
    "low": 228.34,
    "high": 228.72,
    "close": 228.46,
    "volume": 441978
  },
  {
    "date": "2025-09-11 10:40:00",
    "open": 229.2,
    "low": 228.52,
    "high": 229.2,
    "close": 228.64,
    "volume": 447842
  },
  {
    "date": "2025-09-11 10:35:00",
    "open": 229.15,
    "low": 229.04,
    "high": 229.3,
    "close": 229.17,
    "volume": 409249
  },
  {
    "date": "2025-09-11 10:30:00",
    "open": 228.93,
    "low": 228.8,
    "high": 229.2,
    "close": 229.18,
    "volume": 452733
  },
  {
    "date": "2025-09-11 10:25:00",
    "open": 228.85,
    "low": 228.36,
    "high": 228.95,
    "close": 228.93,
    "volume": 518284
  },
  {
    "date": "2025-09-11 10:20:00",
    "open": 228.86,
    "low": 228.78,
    "high": 229.08,
    "close": 228.85,
    "volume": 546664
  },
  {
    "date": "2025-09-11 10:15:00",
    "open": 229.03,
    "low": 228.52,
    "high": 229.11,
    "close": 228.89,
    "volume": 743494
  },
  {
    "date": "2025-09-11 10:10:00",
    "open": 229.11,
    "low": 228.89,
    "high": 229.16,
    "close": 228.98,
    "volume": 939889
  },
  {
    "date": "2025-09-11 10:05:00",
    "open": 229,
    "low": 228.76,
    "high": 229.18,
    "close": 229.09,
    "volume": 894052
  },
  {
    "date": "2025-09-11 10:00:00",
    "open": 229.07,
    "low": 228.78,
    "high": 229.1,
    "close": 228.98,
    "volume": 935214
  },
  {
    "date": "2025-09-11 09:55:00",
    "open": 228.52,
    "low": 228.34,
    "high": 229.14,
    "close": 229.01,
    "volume": 1176000
  },
  {
    "date": "2025-09-11 09:50:00",
    "open": 228.09,
    "low": 227.9,
    "high": 228.52,
    "close": 228.5,
    "volume": 733958
  },
  {
    "date": "2025-09-11 09:45:00",
    "open": 227.92,
    "low": 227.91,
    "high": 228.41,
    "close": 228.09,
    "volume": 2240656
  },
  {
    "date": "2025-09-11 09:40:00",
    "open": 228.27,
    "low": 227.61,
    "high": 228.43,
    "close": 227.95,
    "volume": 950610
  },
  {
    "date": "2025-09-11 09:35:00",
    "open": 227.45,
    "low": 227.45,
    "high": 228.25,
    "close": 228.25,
    "volume": 1210155
  },
  {
    "date": "2025-09-11 09:30:00",
    "open": 226.8,
    "low": 226.65,
    "high": 227.74,
    "close": 227.49,
    "volume": 4302578
  },
  {
    "date": "2025-09-10 15:55:00",
    "open": 226.7,
    "low": 226.5,
    "high": 227.06,
    "close": 226.79,
    "volume": 2702830
  },
  {
    "date": "2025-09-10 15:50:00",
    "open": 226.61,
    "low": 226.6,
    "high": 227.3,
    "close": 226.7,
    "volume": 1587755
  },
  {
    "date": "2025-09-10 15:45:00",
    "open": 226.46,
    "low": 226.39,
    "high": 226.68,
    "close": 226.61,
    "volume": 798981
  },
  {
    "date": "2025-09-10 15:40:00",
    "open": 226.05,
    "low": 225.95,
    "high": 226.5,
    "close": 226.46,
    "volume": 930500
  },
  {
    "date": "2025-09-10 15:35:00",
    "open": 226.16,
    "low": 225.96,
    "high": 226.28,
    "close": 226.04,
    "volume": 633200
  },
  {
    "date": "2025-09-10 15:30:00",
    "open": 226.18,
    "low": 225.96,
    "high": 226.36,
    "close": 226.17,
    "volume": 1074225
  },
  {
    "date": "2025-09-10 15:25:00",
    "open": 226.44,
    "low": 226.19,
    "high": 226.51,
    "close": 226.19,
    "volume": 589253
  },
  {
    "date": "2025-09-10 15:20:00",
    "open": 226.46,
    "low": 226.25,
    "high": 226.57,
    "close": 226.45,
    "volume": 595691
  },
  {
    "date": "2025-09-10 15:15:00",
    "open": 226.29,
    "low": 226.28,
    "high": 226.6,
    "close": 226.47,
    "volume": 471603
  },
  {
    "date": "2025-09-10 15:10:00",
    "open": 226.49,
    "low": 226.26,
    "high": 226.63,
    "close": 226.28,
    "volume": 69124
  },
  {
    "date": "2025-09-10 15:05:00",
    "open": 226.63,
    "low": 226.48,
    "high": 226.7,
    "close": 226.48,
    "volume": 458244
  },
  {
    "date": "2025-09-10 15:00:00",
    "open": 226.89,
    "low": 226.6,
    "high": 227.04,
    "close": 226.63,
    "volume": 641415
  },
  {
    "date": "2025-09-10 14:55:00",
    "open": 226.66,
    "low": 226.6,
    "high": 226.92,
    "close": 226.9,
    "volume": 585077
  },
  {
    "date": "2025-09-10 14:50:00",
    "open": 226.53,
    "low": 226.46,
    "high": 226.7,
    "close": 226.66,
    "volume": 347356
  },
  {
    "date": "2025-09-10 14:45:00",
    "open": 226.82,
    "low": 226.42,
    "high": 226.87,
    "close": 226.53,
    "volume": 530852
  },
  {
    "date": "2025-09-10 14:40:00",
    "open": 227,
    "low": 226.8,
    "high": 227.05,
    "close": 226.82,
    "volume": 308464
  },
  {
    "date": "2025-09-10 14:35:00",
    "open": 227.26,
    "low": 226.95,
    "high": 227.27,
    "close": 227,
    "volume": 520897
  },
  {
    "date": "2025-09-10 14:30:00",
    "open": 227.06,
    "low": 227.06,
    "high": 227.35,
    "close": 227.27,
    "volume": 526431
  },
  {
    "date": "2025-09-10 14:25:00",
    "open": 226.97,
    "low": 226.89,
    "high": 227.08,
    "close": 227.05,
    "volume": 399695
  },
  {
    "date": "2025-09-10 14:20:00",
    "open": 226.68,
    "low": 226.65,
    "high": 227.02,
    "close": 226.98,
    "volume": 347972
  },
  {
    "date": "2025-09-10 14:15:00",
    "open": 226.91,
    "low": 226.63,
    "high": 226.99,
    "close": 226.67,
    "volume": 1313641
  },
  {
    "date": "2025-09-10 14:10:00",
    "open": 226.75,
    "low": 226.73,
    "high": 226.93,
    "close": 226.91,
    "volume": 399506
  },
  {
    "date": "2025-09-10 14:05:00",
    "open": 226.68,
    "low": 226.36,
    "high": 226.74,
    "close": 226.74,
    "volume": 646381
  },
  {
    "date": "2025-09-10 14:00:00",
    "open": 226.68,
    "low": 226.52,
    "high": 226.72,
    "close": 226.68,
    "volume": 446996
  },
  {
    "date": "2025-09-10 13:55:00",
    "open": 226.9,
    "low": 226.59,
    "high": 226.9,
    "close": 226.69,
    "volume": 499335
  },
  {
    "date": "2025-09-10 13:50:00",
    "open": 226.82,
    "low": 226.8,
    "high": 226.99,
    "close": 226.92,
    "volume": 359164
  },
  {
    "date": "2025-09-10 13:45:00",
    "open": 226.68,
    "low": 226.65,
    "high": 226.92,
    "close": 226.83,
    "volume": 366566
  },
  {
    "date": "2025-09-10 13:40:00",
    "open": 226.7,
    "low": 226.63,
    "high": 226.86,
    "close": 226.68,
    "volume": 369381
  },
  {
    "date": "2025-09-10 13:35:00",
    "open": 226.97,
    "low": 226.6,
    "high": 227.08,
    "close": 226.71,
    "volume": 611940
  },
  {
    "date": "2025-09-10 13:30:00",
    "open": 226.81,
    "low": 226.81,
    "high": 227.07,
    "close": 226.97,
    "volume": 454066
  },
  {
    "date": "2025-09-10 13:25:00",
    "open": 226.9,
    "low": 226.63,
    "high": 226.9,
    "close": 226.8,
    "volume": 448114
  },
  {
    "date": "2025-09-10 13:20:00",
    "open": 227.24,
    "low": 226.83,
    "high": 227.26,
    "close": 226.9,
    "volume": 580890
  },
  {
    "date": "2025-09-10 13:15:00",
    "open": 227.49,
    "low": 227.22,
    "high": 227.65,
    "close": 227.23,
    "volume": 356323
  },
  {
    "date": "2025-09-10 13:10:00",
    "open": 227.62,
    "low": 227.44,
    "high": 227.74,
    "close": 227.49,
    "volume": 355921
  },
  {
    "date": "2025-09-10 13:05:00",
    "open": 227.56,
    "low": 227.39,
    "high": 227.63,
    "close": 227.63,
    "volume": 377374
  },
  {
    "date": "2025-09-10 13:00:00",
    "open": 227.41,
    "low": 227.32,
    "high": 227.57,
    "close": 227.56,
    "volume": 453024
  },
  {
    "date": "2025-09-10 12:55:00",
    "open": 227.17,
    "low": 227.1,
    "high": 227.42,
    "close": 227.41,
    "volume": 360399
  },
  {
    "date": "2025-09-10 12:50:00",
    "open": 227.11,
    "low": 227.05,
    "high": 227.32,
    "close": 227.17,
    "volume": 482326
  },
  {
    "date": "2025-09-10 12:45:00",
    "open": 227.02,
    "low": 226.99,
    "high": 227.33,
    "close": 227.1,
    "volume": 527243
  },
  {
    "date": "2025-09-10 12:40:00",
    "open": 226.97,
    "low": 226.82,
    "high": 227.1,
    "close": 227.02,
    "volume": 1555520
  },
  {
    "date": "2025-09-10 12:35:00",
    "open": 227.32,
    "low": 226.98,
    "high": 227.35,
    "close": 226.99,
    "volume": 523850
  },
  {
    "date": "2025-09-10 12:30:00",
    "open": 227.75,
    "low": 227.31,
    "high": 227.79,
    "close": 227.32,
    "volume": 558267
  },
  {
    "date": "2025-09-10 12:25:00",
    "open": 227.95,
    "low": 227.69,
    "high": 228.03,
    "close": 227.74,
    "volume": 693214
  },
  {
    "date": "2025-09-10 12:20:00",
    "open": 227.81,
    "low": 227.62,
    "high": 228.06,
    "close": 227.95,
    "volume": 708472
  },
  {
    "date": "2025-09-10 12:15:00",
    "open": 227.74,
    "low": 227.55,
    "high": 227.86,
    "close": 227.81,
    "volume": 715135
  },
  {
    "date": "2025-09-10 12:10:00",
    "open": 227.28,
    "low": 227.18,
    "high": 227.79,
    "close": 227.75,
    "volume": 708786
  },
  {
    "date": "2025-09-10 12:05:00",
    "open": 227.26,
    "low": 226.99,
    "high": 227.3,
    "close": 227.28,
    "volume": 727110
  },
  {
    "date": "2025-09-10 12:00:00",
    "open": 227.26,
    "low": 227.23,
    "high": 227.47,
    "close": 227.26,
    "volume": 701526
  },
  {
    "date": "2025-09-10 11:55:00",
    "open": 226.99,
    "low": 226.99,
    "high": 227.35,
    "close": 227.26,
    "volume": 775666
  },
  {
    "date": "2025-09-10 11:50:00",
    "open": 227.2,
    "low": 226.74,
    "high": 227.21,
    "close": 227,
    "volume": 849216
  },
  {
    "date": "2025-09-10 11:45:00",
    "open": 226.79,
    "low": 226.73,
    "high": 227.23,
    "close": 227.2,
    "volume": 807035
  },
  {
    "date": "2025-09-10 11:40:00",
    "open": 226.45,
    "low": 226.24,
    "high": 226.89,
    "close": 226.76,
    "volume": 1034840
  },
  {
    "date": "2025-09-10 11:35:00",
    "open": 226.66,
    "low": 226.37,
    "high": 226.74,
    "close": 226.46,
    "volume": 1115448
  },
  {
    "date": "2025-09-10 11:30:00",
    "open": 227.05,
    "low": 226.54,
    "high": 227.1,
    "close": 226.65,
    "volume": 1320413
  },
  {
    "date": "2025-09-10 11:25:00",
    "open": 227.33,
    "low": 227.02,
    "high": 227.35,
    "close": 227.03,
    "volume": 769590
  },
  {
    "date": "2025-09-10 11:20:00",
    "open": 227.14,
    "low": 227.05,
    "high": 227.66,
    "close": 227.32,
    "volume": 968201
  },
  {
    "date": "2025-09-10 11:15:00",
    "open": 227.86,
    "low": 227.14,
    "high": 227.89,
    "close": 227.16,
    "volume": 1140762
  },
  {
    "date": "2025-09-10 11:10:00",
    "open": 227.81,
    "low": 227.81,
    "high": 228.09,
    "close": 227.86,
    "volume": 1132568
  },
  {
    "date": "2025-09-10 11:05:00",
    "open": 227.63,
    "low": 227.43,
    "high": 227.81,
    "close": 227.81,
    "volume": 945564
  },
  {
    "date": "2025-09-10 11:00:00",
    "open": 227.74,
    "low": 227.58,
    "high": 227.81,
    "close": 227.63,
    "volume": 749733
  },
  {
    "date": "2025-09-10 10:55:00",
    "open": 228.02,
    "low": 227.73,
    "high": 228.25,
    "close": 227.74,
    "volume": 1005187
  },
  {
    "date": "2025-09-10 10:50:00",
    "open": 227.37,
    "low": 227.37,
    "high": 228.05,
    "close": 228.04,
    "volume": 981741
  },
  {
    "date": "2025-09-10 10:45:00",
    "open": 227.78,
    "low": 227.36,
    "high": 227.86,
    "close": 227.36,
    "volume": 1187227
  },
  {
    "date": "2025-09-10 10:40:00",
    "open": 228.14,
    "low": 227.72,
    "high": 228.25,
    "close": 227.81,
    "volume": 871851
  },
  {
    "date": "2025-09-10 10:35:00",
    "open": 228.08,
    "low": 227.71,
    "high": 228.25,
    "close": 228.14,
    "volume": 964012
  },
  {
    "date": "2025-09-10 10:30:00",
    "open": 228.1,
    "low": 227.85,
    "high": 228.33,
    "close": 228.07,
    "volume": 844110
  },
  {
    "date": "2025-09-10 10:25:00",
    "open": 227.78,
    "low": 227.55,
    "high": 228.42,
    "close": 228.11,
    "volume": 1419272
  },
  {
    "date": "2025-09-10 10:20:00",
    "open": 228.59,
    "low": 227.64,
    "high": 228.59,
    "close": 227.77,
    "volume": 1777026
  },
  {
    "date": "2025-09-10 10:15:00",
    "open": 229.22,
    "low": 228.34,
    "high": 229.4,
    "close": 228.58,
    "volume": 1413460
  },
  {
    "date": "2025-09-10 10:10:00",
    "open": 228.99,
    "low": 228.97,
    "high": 229.65,
    "close": 229.24,
    "volume": 1120757
  },
  {
    "date": "2025-09-10 10:05:00",
    "open": 228.89,
    "low": 228.74,
    "high": 229.15,
    "close": 228.98,
    "volume": 1091340
  },
  {
    "date": "2025-09-10 10:00:00",
    "open": 229.54,
    "low": 228.68,
    "high": 229.78,
    "close": 228.94,
    "volume": 1478087
  },
  {
    "date": "2025-09-10 09:55:00",
    "open": 230.16,
    "low": 229.37,
    "high": 230.3,
    "close": 229.53,
    "volume": 1165826
  },
  {
    "date": "2025-09-10 09:50:00",
    "open": 229.94,
    "low": 229.6,
    "high": 230.53,
    "close": 230.17,
    "volume": 1415209
  },
  {
    "date": "2025-09-10 09:45:00",
    "open": 229.23,
    "low": 229.15,
    "high": 230.16,
    "close": 229.93,
    "volume": 1416102
  },
  {
    "date": "2025-09-10 09:40:00",
    "open": 229.77,
    "low": 228.85,
    "high": 230.19,
    "close": 229.18,
    "volume": 2003111
  },
  {
    "date": "2025-09-10 09:35:00",
    "open": 230.61,
    "low": 229.69,
    "high": 230.83,
    "close": 229.81,
    "volume": 2613727
  },
  {
    "date": "2025-09-10 09:30:00",
    "open": 232.21,
    "low": 230.39,
    "high": 232.42,
    "close": 230.63,
    "volume": 5253318
  },
  {
    "date": "2025-09-09 15:55:00",
    "open": 234.5,
    "low": 234.26,
    "high": 234.59,
    "close": 234.35,
    "volume": 2073474
  },
  {
    "date": "2025-09-09 15:50:00",
    "open": 234.33,
    "low": 233.98,
    "high": 234.64,
    "close": 234.49,
    "volume": 1387974
  },
  {
    "date": "2025-09-09 15:45:00",
    "open": 233.57,
    "low": 233.52,
    "high": 234,
    "close": 233.96,
    "volume": 882274
  },
  {
    "date": "2025-09-09 15:40:00",
    "open": 234.29,
    "low": 233.52,
    "high": 234.29,
    "close": 233.59,
    "volume": 912958
  },
  {
    "date": "2025-09-09 15:35:00",
    "open": 234.32,
    "low": 234.1,
    "high": 234.36,
    "close": 234.29,
    "volume": 499307
  },
  {
    "date": "2025-09-09 15:30:00",
    "open": 234.31,
    "low": 234.17,
    "high": 234.43,
    "close": 234.34,
    "volume": 561192
  },
  {
    "date": "2025-09-09 15:25:00",
    "open": 234.38,
    "low": 234.29,
    "high": 234.49,
    "close": 234.32,
    "volume": 439737
  },
  {
    "date": "2025-09-09 15:20:00",
    "open": 234.31,
    "low": 234.3,
    "high": 234.49,
    "close": 234.39,
    "volume": 519118
  },
  {
    "date": "2025-09-09 15:15:00",
    "open": 234.24,
    "low": 234.23,
    "high": 234.48,
    "close": 234.3,
    "volume": 706423
  },
  {
    "date": "2025-09-09 15:10:00",
    "open": 233.95,
    "low": 233.87,
    "high": 234.27,
    "close": 234.23,
    "volume": 672092
  },
  {
    "date": "2025-09-09 15:05:00",
    "open": 233.72,
    "low": 233.69,
    "high": 234,
    "close": 233.95,
    "volume": 753881
  },
  {
    "date": "2025-09-09 15:00:00",
    "open": 233.74,
    "low": 233.56,
    "high": 234.04,
    "close": 233.74,
    "volume": 1000527
  },
  {
    "date": "2025-09-09 14:55:00",
    "open": 233.73,
    "low": 233.36,
    "high": 233.8,
    "close": 233.73,
    "volume": 800940
  },
  {
    "date": "2025-09-09 14:50:00",
    "open": 233.71,
    "low": 233.61,
    "high": 233.94,
    "close": 233.67,
    "volume": 803129
  },
  {
    "date": "2025-09-09 14:45:00",
    "open": 233.81,
    "low": 233.38,
    "high": 233.88,
    "close": 233.72,
    "volume": 1214243
  },
  {
    "date": "2025-09-09 14:40:00",
    "open": 233.9,
    "low": 233.79,
    "high": 234.35,
    "close": 233.81,
    "volume": 903054
  },
  {
    "date": "2025-09-09 14:35:00",
    "open": 234.23,
    "low": 233.88,
    "high": 234.4,
    "close": 233.9,
    "volume": 887608
  },
  {
    "date": "2025-09-09 14:30:00",
    "open": 234.57,
    "low": 234.05,
    "high": 234.7,
    "close": 234.22,
    "volume": 843866
  },
  {
    "date": "2025-09-09 14:25:00",
    "open": 234.83,
    "low": 234.38,
    "high": 234.94,
    "close": 234.54,
    "volume": 942977
  },
  {
    "date": "2025-09-09 14:20:00",
    "open": 234.44,
    "low": 234.21,
    "high": 234.92,
    "close": 234.81,
    "volume": 889672
  },
  {
    "date": "2025-09-09 14:15:00",
    "open": 235.04,
    "low": 233.84,
    "high": 235.1,
    "close": 234.46,
    "volume": 1976577
  },
  {
    "date": "2025-09-09 14:10:00",
    "open": 235.83,
    "low": 233.66,
    "high": 235.95,
    "close": 235.08,
    "volume": 4546571
  },
  {
    "date": "2025-09-09 14:05:00",
    "open": 236.29,
    "low": 235.78,
    "high": 236.33,
    "close": 235.81,
    "volume": 901793
  },
  {
    "date": "2025-09-09 14:00:00",
    "open": 236.51,
    "low": 236.25,
    "high": 236.76,
    "close": 236.3,
    "volume": 457059
  },
  {
    "date": "2025-09-09 13:55:00",
    "open": 236.62,
    "low": 236.43,
    "high": 236.89,
    "close": 236.53,
    "volume": 492076
  },
  {
    "date": "2025-09-09 13:50:00",
    "open": 236.55,
    "low": 236.43,
    "high": 237,
    "close": 236.63,
    "volume": 705233
  },
  {
    "date": "2025-09-09 13:45:00",
    "open": 236.81,
    "low": 236.44,
    "high": 237.51,
    "close": 236.55,
    "volume": 883738
  },
  {
    "date": "2025-09-09 13:40:00",
    "open": 237.02,
    "low": 236.32,
    "high": 237.42,
    "close": 236.82,
    "volume": 1517616
  },
  {
    "date": "2025-09-09 13:35:00",
    "open": 237.15,
    "low": 236.9,
    "high": 237.9,
    "close": 237.07,
    "volume": 1255752
  },
  {
    "date": "2025-09-09 13:30:00",
    "open": 237.75,
    "low": 236.85,
    "high": 238.09,
    "close": 237.19,
    "volume": 1505612
  },
  {
    "date": "2025-09-09 13:25:00",
    "open": 237.7,
    "low": 237.64,
    "high": 238.22,
    "close": 237.75,
    "volume": 830489
  },
  {
    "date": "2025-09-09 13:20:00",
    "open": 238.02,
    "low": 237.35,
    "high": 238.2,
    "close": 237.69,
    "volume": 1128704
  },
  {
    "date": "2025-09-09 13:15:00",
    "open": 238.05,
    "low": 237.9,
    "high": 238.78,
    "close": 238.01,
    "volume": 1421764
  },
  {
    "date": "2025-09-09 13:10:00",
    "open": 237.4,
    "low": 237.33,
    "high": 238.57,
    "close": 238.05,
    "volume": 214714
  },
  {
    "date": "2025-09-09 13:05:00",
    "open": 236.97,
    "low": 236.77,
    "high": 237.4,
    "close": 237.39,
    "volume": 1226466
  },
  {
    "date": "2025-09-09 13:00:00",
    "open": 236.62,
    "low": 236.62,
    "high": 237.15,
    "close": 236.98,
    "volume": 1025426
  },
  {
    "date": "2025-09-09 12:55:00",
    "open": 236.26,
    "low": 236.26,
    "high": 236.67,
    "close": 236.61,
    "volume": 410884
  },
  {
    "date": "2025-09-09 12:50:00",
    "open": 236.37,
    "low": 236.23,
    "high": 236.42,
    "close": 236.25,
    "volume": 319329
  },
  {
    "date": "2025-09-09 12:45:00",
    "open": 235.97,
    "low": 235.95,
    "high": 236.4,
    "close": 236.35,
    "volume": 355336
  },
  {
    "date": "2025-09-09 12:40:00",
    "open": 235.89,
    "low": 235.84,
    "high": 235.99,
    "close": 235.98,
    "volume": 214477
  },
  {
    "date": "2025-09-09 12:35:00",
    "open": 236.1,
    "low": 235.86,
    "high": 236.11,
    "close": 235.91,
    "volume": 194848
  },
  {
    "date": "2025-09-09 12:30:00",
    "open": 236,
    "low": 235.83,
    "high": 236.12,
    "close": 236.11,
    "volume": 222621
  },
  {
    "date": "2025-09-09 12:25:00",
    "open": 235.84,
    "low": 235.83,
    "high": 236.06,
    "close": 235.99,
    "volume": 287947
  },
  {
    "date": "2025-09-09 12:20:00",
    "open": 235.93,
    "low": 235.68,
    "high": 235.95,
    "close": 235.83,
    "volume": 322340
  },
  {
    "date": "2025-09-09 12:15:00",
    "open": 236.05,
    "low": 235.91,
    "high": 236.22,
    "close": 235.95,
    "volume": 360907
  },
  {
    "date": "2025-09-09 12:10:00",
    "open": 235.98,
    "low": 235.84,
    "high": 236.09,
    "close": 236.05,
    "volume": 516821
  },
  {
    "date": "2025-09-09 12:05:00",
    "open": 235.8,
    "low": 235.72,
    "high": 235.97,
    "close": 235.97,
    "volume": 226616
  },
  {
    "date": "2025-09-09 12:00:00",
    "open": 235.67,
    "low": 235.56,
    "high": 235.86,
    "close": 235.79,
    "volume": 377446
  },
  {
    "date": "2025-09-09 11:55:00",
    "open": 236.13,
    "low": 235.56,
    "high": 236.14,
    "close": 235.67,
    "volume": 700122
  },
  {
    "date": "2025-09-09 11:50:00",
    "open": 235.79,
    "low": 235.79,
    "high": 236.15,
    "close": 236.08,
    "volume": 450971
  },
  {
    "date": "2025-09-09 11:45:00",
    "open": 235.95,
    "low": 235.76,
    "high": 236.1,
    "close": 235.78,
    "volume": 289743
  },
  {
    "date": "2025-09-09 11:40:00",
    "open": 236.16,
    "low": 235.92,
    "high": 236.16,
    "close": 235.94,
    "volume": 268081
  },
  {
    "date": "2025-09-09 11:35:00",
    "open": 236.1,
    "low": 236.03,
    "high": 236.2,
    "close": 236.16,
    "volume": 213753
  },
  {
    "date": "2025-09-09 11:30:00",
    "open": 236.22,
    "low": 236.01,
    "high": 236.26,
    "close": 236.09,
    "volume": 285383
  },
  {
    "date": "2025-09-09 11:25:00",
    "open": 236.39,
    "low": 236.16,
    "high": 236.4,
    "close": 236.23,
    "volume": 233940
  },
  {
    "date": "2025-09-09 11:20:00",
    "open": 236.26,
    "low": 236.18,
    "high": 236.4,
    "close": 236.37,
    "volume": 273830
  },
  {
    "date": "2025-09-09 11:15:00",
    "open": 236.16,
    "low": 236,
    "high": 236.33,
    "close": 236.26,
    "volume": 369869
  },
  {
    "date": "2025-09-09 11:10:00",
    "open": 236.09,
    "low": 236.03,
    "high": 236.16,
    "close": 236.15,
    "volume": 272992
  },
  {
    "date": "2025-09-09 11:05:00",
    "open": 236.54,
    "low": 235.94,
    "high": 236.55,
    "close": 236.1,
    "volume": 403113
  },
  {
    "date": "2025-09-09 11:00:00",
    "open": 236.54,
    "low": 236.29,
    "high": 236.68,
    "close": 236.54,
    "volume": 305671
  },
  {
    "date": "2025-09-09 10:55:00",
    "open": 236.55,
    "low": 236.45,
    "high": 236.67,
    "close": 236.54,
    "volume": 269154
  },
  {
    "date": "2025-09-09 10:50:00",
    "open": 236.42,
    "low": 236.23,
    "high": 236.67,
    "close": 236.53,
    "volume": 364181
  },
  {
    "date": "2025-09-09 10:45:00",
    "open": 235.86,
    "low": 235.78,
    "high": 236.49,
    "close": 236.43,
    "volume": 432005
  },
  {
    "date": "2025-09-09 10:40:00",
    "open": 235.92,
    "low": 235.8,
    "high": 236.04,
    "close": 235.88,
    "volume": 518910
  },
  {
    "date": "2025-09-09 10:35:00",
    "open": 236.04,
    "low": 235.92,
    "high": 236.17,
    "close": 235.92,
    "volume": 463713
  },
  {
    "date": "2025-09-09 10:30:00",
    "open": 236.3,
    "low": 236.04,
    "high": 236.35,
    "close": 236.04,
    "volume": 370097
  },
  {
    "date": "2025-09-09 10:25:00",
    "open": 236.55,
    "low": 236.3,
    "high": 236.57,
    "close": 236.31,
    "volume": 268194
  },
  {
    "date": "2025-09-09 10:20:00",
    "open": 236.31,
    "low": 236.12,
    "high": 236.58,
    "close": 236.56,
    "volume": 384874
  },
  {
    "date": "2025-09-09 10:15:00",
    "open": 236.44,
    "low": 236.08,
    "high": 236.59,
    "close": 236.33,
    "volume": 535646
  },
  {
    "date": "2025-09-09 10:10:00",
    "open": 236.65,
    "low": 236.42,
    "high": 236.78,
    "close": 236.43,
    "volume": 578686
  },
  {
    "date": "2025-09-09 10:05:00",
    "open": 236.71,
    "low": 236.42,
    "high": 236.88,
    "close": 236.63,
    "volume": 416640
  },
  {
    "date": "2025-09-09 10:00:00",
    "open": 236.8,
    "low": 236.34,
    "high": 236.91,
    "close": 236.71,
    "volume": 558677
  },
  {
    "date": "2025-09-09 09:55:00",
    "open": 236.54,
    "low": 236.45,
    "high": 237.01,
    "close": 236.8,
    "volume": 589525
  },
  {
    "date": "2025-09-09 09:50:00",
    "open": 236.46,
    "low": 236.22,
    "high": 236.59,
    "close": 236.53,
    "volume": 541352
  },
  {
    "date": "2025-09-09 09:45:00",
    "open": 236.69,
    "low": 236.42,
    "high": 236.85,
    "close": 236.47,
    "volume": 642920
  },
  {
    "date": "2025-09-09 09:40:00",
    "open": 236.47,
    "low": 236.35,
    "high": 236.96,
    "close": 236.66,
    "volume": 653085
  },
  {
    "date": "2025-09-09 09:35:00",
    "open": 237.79,
    "low": 236.47,
    "high": 237.84,
    "close": 236.48,
    "volume": 754152
  },
  {
    "date": "2025-09-09 09:30:00",
    "open": 236.99,
    "low": 236.74,
    "high": 238.01,
    "close": 237.76,
    "volume": 1967001
  },
  {
    "date": "2025-09-08 15:55:00",
    "open": 237.59,
    "low": 237.44,
    "high": 237.97,
    "close": 237.87,
    "volume": 2619318
  },
  {
    "date": "2025-09-08 15:50:00",
    "open": 237.75,
    "low": 237.56,
    "high": 237.87,
    "close": 237.58,
    "volume": 795942
  },
  {
    "date": "2025-09-08 15:45:00",
    "open": 237.5,
    "low": 237.49,
    "high": 237.76,
    "close": 237.74,
    "volume": 506802
  },
  {
    "date": "2025-09-08 15:40:00",
    "open": 237.22,
    "low": 237.06,
    "high": 237.51,
    "close": 237.48,
    "volume": 540738
  },
  {
    "date": "2025-09-08 15:35:00",
    "open": 237.02,
    "low": 236.91,
    "high": 237.22,
    "close": 237.19,
    "volume": 351002
  },
  {
    "date": "2025-09-08 15:30:00",
    "open": 237.12,
    "low": 237.01,
    "high": 237.31,
    "close": 237.02,
    "volume": 363397
  },
  {
    "date": "2025-09-08 15:25:00",
    "open": 237.02,
    "low": 236.96,
    "high": 237.14,
    "close": 237.11,
    "volume": 288660
  },
  {
    "date": "2025-09-08 15:20:00",
    "open": 237.1,
    "low": 236.84,
    "high": 237.2,
    "close": 237.01,
    "volume": 425972
  },
  {
    "date": "2025-09-08 15:15:00",
    "open": 236.8,
    "low": 236.75,
    "high": 237.11,
    "close": 237.1,
    "volume": 323896
  },
  {
    "date": "2025-09-08 15:10:00",
    "open": 236.97,
    "low": 236.75,
    "high": 237.01,
    "close": 236.81,
    "volume": 271480
  },
  {
    "date": "2025-09-08 15:05:00",
    "open": 236.86,
    "low": 236.86,
    "high": 237.09,
    "close": 236.97,
    "volume": 263622
  },
  {
    "date": "2025-09-08 15:00:00",
    "open": 236.88,
    "low": 236.8,
    "high": 236.91,
    "close": 236.86,
    "volume": 228447
  },
  {
    "date": "2025-09-08 14:55:00",
    "open": 236.65,
    "low": 236.63,
    "high": 236.88,
    "close": 236.88,
    "volume": 395447
  },
  {
    "date": "2025-09-08 14:50:00",
    "open": 236.85,
    "low": 236.61,
    "high": 236.86,
    "close": 236.65,
    "volume": 266898
  },
  {
    "date": "2025-09-08 14:45:00",
    "open": 236.87,
    "low": 236.79,
    "high": 237.02,
    "close": 236.85,
    "volume": 329815
  },
  {
    "date": "2025-09-08 14:40:00",
    "open": 236.82,
    "low": 236.65,
    "high": 236.9,
    "close": 236.87,
    "volume": 403680
  },
  {
    "date": "2025-09-08 14:35:00",
    "open": 237.03,
    "low": 236.8,
    "high": 237.05,
    "close": 236.81,
    "volume": 271753
  },
  {
    "date": "2025-09-08 14:30:00",
    "open": 237.16,
    "low": 236.94,
    "high": 237.19,
    "close": 237.04,
    "volume": 291745
  },
  {
    "date": "2025-09-08 14:25:00",
    "open": 237.26,
    "low": 237.11,
    "high": 237.39,
    "close": 237.16,
    "volume": 294560
  },
  {
    "date": "2025-09-08 14:20:00",
    "open": 237.36,
    "low": 237.27,
    "high": 237.4,
    "close": 237.27,
    "volume": 210068
  },
  {
    "date": "2025-09-08 14:15:00",
    "open": 237.78,
    "low": 237.28,
    "high": 237.79,
    "close": 237.34,
    "volume": 267256
  },
  {
    "date": "2025-09-08 14:10:00",
    "open": 237.57,
    "low": 237.45,
    "high": 237.84,
    "close": 237.82,
    "volume": 439745
  },
  {
    "date": "2025-09-08 14:05:00",
    "open": 237.7,
    "low": 237.46,
    "high": 237.74,
    "close": 237.58,
    "volume": 538117
  },
  {
    "date": "2025-09-08 14:00:00",
    "open": 237.76,
    "low": 237.56,
    "high": 237.79,
    "close": 237.73,
    "volume": 343628
  },
  {
    "date": "2025-09-08 13:55:00",
    "open": 237.87,
    "low": 237.68,
    "high": 237.95,
    "close": 237.75,
    "volume": 288145
  },
  {
    "date": "2025-09-08 13:50:00",
    "open": 237.63,
    "low": 237.63,
    "high": 237.9,
    "close": 237.89,
    "volume": 447108
  },
  {
    "date": "2025-09-08 13:45:00",
    "open": 237.87,
    "low": 237.58,
    "high": 237.95,
    "close": 237.61,
    "volume": 255087
  },
  {
    "date": "2025-09-08 13:40:00",
    "open": 237.66,
    "low": 237.5,
    "high": 237.87,
    "close": 237.87,
    "volume": 350869
  },
  {
    "date": "2025-09-08 13:35:00",
    "open": 237.61,
    "low": 237.61,
    "high": 237.83,
    "close": 237.66,
    "volume": 254287
  },
  {
    "date": "2025-09-08 13:30:00",
    "open": 237.8,
    "low": 237.59,
    "high": 237.91,
    "close": 237.65,
    "volume": 425409
  },
  {
    "date": "2025-09-08 13:25:00",
    "open": 237.47,
    "low": 237.47,
    "high": 237.87,
    "close": 237.8,
    "volume": 497574
  },
  {
    "date": "2025-09-08 13:20:00",
    "open": 237.05,
    "low": 237.04,
    "high": 237.51,
    "close": 237.45,
    "volume": 513337
  },
  {
    "date": "2025-09-08 13:15:00",
    "open": 236.43,
    "low": 236.38,
    "high": 237.07,
    "close": 237.05,
    "volume": 441439
  },
  {
    "date": "2025-09-08 13:10:00",
    "open": 236.61,
    "low": 236.35,
    "high": 236.76,
    "close": 236.41,
    "volume": 349571
  },
  {
    "date": "2025-09-08 13:05:00",
    "open": 236.64,
    "low": 236.42,
    "high": 236.78,
    "close": 236.61,
    "volume": 456822
  },
  {
    "date": "2025-09-08 13:00:00",
    "open": 236.98,
    "low": 236.58,
    "high": 237.08,
    "close": 236.63,
    "volume": 395253
  },
  {
    "date": "2025-09-08 12:55:00",
    "open": 237.2,
    "low": 236.86,
    "high": 237.22,
    "close": 236.98,
    "volume": 315381
  },
  {
    "date": "2025-09-08 12:50:00",
    "open": 237.19,
    "low": 236.97,
    "high": 237.28,
    "close": 237.19,
    "volume": 332561
  },
  {
    "date": "2025-09-08 12:45:00",
    "open": 237.45,
    "low": 236.94,
    "high": 237.45,
    "close": 237.18,
    "volume": 465192
  },
  {
    "date": "2025-09-08 12:40:00",
    "open": 237.36,
    "low": 237.28,
    "high": 237.53,
    "close": 237.45,
    "volume": 29004
  },
  {
    "date": "2025-09-08 12:35:00",
    "open": 237.51,
    "low": 237.36,
    "high": 237.53,
    "close": 237.36,
    "volume": 237871
  },
  {
    "date": "2025-09-08 12:30:00",
    "open": 237.72,
    "low": 237.37,
    "high": 237.79,
    "close": 237.5,
    "volume": 398632
  },
  {
    "date": "2025-09-08 12:25:00",
    "open": 237.7,
    "low": 237.62,
    "high": 237.8,
    "close": 237.73,
    "volume": 250148
  },
  {
    "date": "2025-09-08 12:20:00",
    "open": 237.78,
    "low": 237.61,
    "high": 237.78,
    "close": 237.69,
    "volume": 456689
  },
  {
    "date": "2025-09-08 12:15:00",
    "open": 237.89,
    "low": 237.71,
    "high": 237.89,
    "close": 237.78,
    "volume": 270164
  },
  {
    "date": "2025-09-08 12:10:00",
    "open": 237.95,
    "low": 237.81,
    "high": 238.03,
    "close": 237.9,
    "volume": 283770
  },
  {
    "date": "2025-09-08 12:05:00",
    "open": 238,
    "low": 237.85,
    "high": 238.09,
    "close": 237.94,
    "volume": 381491
  },
  {
    "date": "2025-09-08 12:00:00",
    "open": 238.29,
    "low": 237.99,
    "high": 238.37,
    "close": 238,
    "volume": 447245
  },
  {
    "date": "2025-09-08 11:55:00",
    "open": 238.21,
    "low": 238.11,
    "high": 238.43,
    "close": 238.28,
    "volume": 393211
  },
  {
    "date": "2025-09-08 11:50:00",
    "open": 238.13,
    "low": 238.08,
    "high": 238.28,
    "close": 238.21,
    "volume": 402585
  },
  {
    "date": "2025-09-08 11:45:00",
    "open": 238.54,
    "low": 237.99,
    "high": 238.54,
    "close": 238.15,
    "volume": 497173
  },
  {
    "date": "2025-09-08 11:40:00",
    "open": 238.44,
    "low": 238.44,
    "high": 238.63,
    "close": 238.55,
    "volume": 273967
  },
  {
    "date": "2025-09-08 11:35:00",
    "open": 238.52,
    "low": 238.35,
    "high": 238.58,
    "close": 238.43,
    "volume": 337534
  },
  {
    "date": "2025-09-08 11:30:00",
    "open": 238.68,
    "low": 238.44,
    "high": 238.77,
    "close": 238.52,
    "volume": 454876
  },
  {
    "date": "2025-09-08 11:25:00",
    "open": 238.79,
    "low": 238.64,
    "high": 238.82,
    "close": 238.7,
    "volume": 324459
  },
  {
    "date": "2025-09-08 11:20:00",
    "open": 238.82,
    "low": 238.62,
    "high": 238.96,
    "close": 238.8,
    "volume": 478759
  },
  {
    "date": "2025-09-08 11:15:00",
    "open": 239.08,
    "low": 238.74,
    "high": 239.09,
    "close": 238.82,
    "volume": 503961
  },
  {
    "date": "2025-09-08 11:10:00",
    "open": 239.07,
    "low": 239.05,
    "high": 239.37,
    "close": 239.12,
    "volume": 333355
  },
  {
    "date": "2025-09-08 11:05:00",
    "open": 239.29,
    "low": 239.02,
    "high": 239.57,
    "close": 239.06,
    "volume": 647395
  },
  {
    "date": "2025-09-08 11:00:00",
    "open": 239.13,
    "low": 239.12,
    "high": 239.37,
    "close": 239.28,
    "volume": 433227
  },
  {
    "date": "2025-09-08 10:55:00",
    "open": 239.05,
    "low": 239.02,
    "high": 239.28,
    "close": 239.15,
    "volume": 522497
  },
  {
    "date": "2025-09-08 10:50:00",
    "open": 239.07,
    "low": 238.84,
    "high": 239.11,
    "close": 239.06,
    "volume": 352283
  },
  {
    "date": "2025-09-08 10:45:00",
    "open": 239.1,
    "low": 238.97,
    "high": 239.28,
    "close": 239.12,
    "volume": 459023
  },
  {
    "date": "2025-09-08 10:40:00",
    "open": 239.12,
    "low": 239.07,
    "high": 239.27,
    "close": 239.09,
    "volume": 374809
  },
  {
    "date": "2025-09-08 10:35:00",
    "open": 239.24,
    "low": 239.08,
    "high": 239.35,
    "close": 239.13,
    "volume": 299849
  },
  {
    "date": "2025-09-08 10:30:00",
    "open": 239.21,
    "low": 239.18,
    "high": 239.49,
    "close": 239.22,
    "volume": 519139
  },
  {
    "date": "2025-09-08 10:25:00",
    "open": 239.24,
    "low": 239.05,
    "high": 239.43,
    "close": 239.23,
    "volume": 395599
  },
  {
    "date": "2025-09-08 10:20:00",
    "open": 239.64,
    "low": 239.21,
    "high": 239.64,
    "close": 239.22,
    "volume": 368015
  },
  {
    "date": "2025-09-08 10:15:00",
    "open": 239.68,
    "low": 239.53,
    "high": 239.84,
    "close": 239.64,
    "volume": 520631
  },
  {
    "date": "2025-09-08 10:10:00",
    "open": 239.43,
    "low": 239.41,
    "high": 239.81,
    "close": 239.68,
    "volume": 715816
  },
  {
    "date": "2025-09-08 10:05:00",
    "open": 239.42,
    "low": 239.3,
    "high": 239.54,
    "close": 239.44,
    "volume": 536277
  },
  {
    "date": "2025-09-08 10:00:00",
    "open": 239.31,
    "low": 239.01,
    "high": 239.42,
    "close": 239.42,
    "volume": 1973369
  },
  {
    "date": "2025-09-08 09:55:00",
    "open": 239.52,
    "low": 239.22,
    "high": 239.6,
    "close": 239.36,
    "volume": 581555
  },
  {
    "date": "2025-09-08 09:50:00",
    "open": 239.29,
    "low": 239.25,
    "high": 239.74,
    "close": 239.53,
    "volume": 792910
  },
  {
    "date": "2025-09-08 09:45:00",
    "open": 239.57,
    "low": 238.92,
    "high": 239.88,
    "close": 239.29,
    "volume": 847637
  },
  {
    "date": "2025-09-08 09:40:00",
    "open": 239.64,
    "low": 239.27,
    "high": 239.7,
    "close": 239.56,
    "volume": 764310
  },
  {
    "date": "2025-09-08 09:35:00",
    "open": 239.74,
    "low": 239.35,
    "high": 240.15,
    "close": 239.66,
    "volume": 1398156
  },
  {
    "date": "2025-09-08 09:30:00",
    "open": 239.38,
    "low": 238.75,
    "high": 239.78,
    "close": 239.72,
    "volume": 2982053
  }
]

In [16]:
type(json_data)

list

In [17]:
len(json_data)

780

In [112]:
# ...existing code...
def json_to_ohlcv_lists_polars(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="datetime", profile: bool = False):
    """
    Load json records into a polars DataFrame and return columns as lists.
    time_type: "datetime" -> Python datetimes, "timestamp" -> int seconds, "str" -> original strings.
    If profile=True an extra key "_timings" is added to the result with per-step timings in milliseconds.
    """
    import time
    start_total = time.perf_counter_ns()

    t0 = time.perf_counter_ns()
    df = pl.DataFrame(json_data)
    t_df = time.perf_counter_ns() - t0

    t1 = time.perf_counter_ns()
    parsed = False
    if "date" in df.columns and df["date"].dtype == pl.Utf8:
        # try 'format' kw for compatibility, fallback to positional arg if signature differs
        try:
            df = df.with_columns(pl.col("date").str.strptime(pl.Datetime, format=time_fmt).alias("date"))
        except TypeError:
            df = df.with_columns(pl.col("date").str.strptime(pl.Datetime, time_fmt).alias("date"))
        parsed = True
    t_parse = time.perf_counter_ns() - t1

    t2 = time.perf_counter_ns()
    if time_type == "str" or "date" not in df.columns:
        times = df["date"].to_list() if "date" in df.columns else [None] * df.height
    elif time_type == "timestamp":
        # polars stores datetimes as ns ints under the hood -> integer seconds via integer division
        times = (df["date"].cast(pl.Int64) // 1_000_000_000).to_list()
    else:  # "datetime"
        # to_list() yields Python datetime objects for Datetime column
        times = df["date"].to_list()
    t_times = time.perf_counter_ns() - t2

    t3 = time.perf_counter_ns()
    opens = df["open"].to_list() if "open" in df.columns else [None] * df.height
    highs = df["high"].to_list() if "high" in df.columns else [None] * df.height
    lows = df["low"].to_list() if "low" in df.columns else [None] * df.height
    closes = df["close"].to_list() if "close" in df.columns else [None] * df.height
    volume = df["volume"].to_list() if "volume" in df.columns else [None] * df.height
    t_cols = time.perf_counter_ns() - t3

    total_ns = time.perf_counter_ns() - start_total

    result = {
        "open": opens,
        "high": highs,
        "low": lows,
        "close": closes,
        "time": times,
        "volume": volume,
    }

    if profile:
        # convert ns -> ms
        result["_timings"] = {
            "total_ms": round(total_ns / 1_000_000, 6),
            "df_construct_ms": round(t_df / 1_000_000, 6),
            "date_parse_ms": round(t_parse / 1_000_000, 6),
            "time_convert_ms": round(t_times / 1_000_000, 6),
            "cols_extract_ms": round(t_cols / 1_000_000, 6),
            "parsed_date_column": parsed,
        }

    return result
# ...existing code...

In [39]:
# ...existing code...
def json_to_ohlcv_lists_py(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="datetime", profile: bool = False):
    """
    Pure-Python extractor mirroring the polars version. Returns the same keys and when
    profile=True an extra "_timings" key with simple per-step timings in milliseconds.
    """
    import time
    start_total = time.perf_counter_ns()

    opens, highs, lows, closes, vols = [], [], [], [], []
    times = []
    parsed_any = False
    parse_ns = 0

    t_loop_start = time.perf_counter_ns()
    for r in json_data:
        opens.append(r.get("open"))
        highs.append(r.get("high"))
        lows.append(r.get("low"))
        closes.append(r.get("close"))
        vols.append(r.get("volume"))
        if "date" in r:
            if time_type == "str":
                times.append(r["date"])
            else:
                t_parse_start = time.perf_counter_ns()
                dt = datetime.strptime(r["date"], time_fmt)
                t_parse_end = time.perf_counter_ns()
                parse_ns += (t_parse_end - t_parse_start)
                parsed_any = True
                if time_type == "timestamp":
                    times.append(int(dt.replace(tzinfo=None).timestamp()))
                else:
                    times.append(dt)
        else:
            times.append(None)
    t_loop = time.perf_counter_ns() - t_loop_start
    total_ns = time.perf_counter_ns() - start_total

    result = {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}

    if profile:
        result["_timings"] = {
            "total_ms": round(total_ns / 1_000_000, 6),
            "loop_ms": round(t_loop / 1_000_000, 6),
            "date_parse_ms": round(parse_ns / 1_000_000, 6),
            "parsed_date_column": parsed_any,
        }

    return result
# ...existing code...

In [54]:
# ...existing code...
def json_to_ohlcv_lists_py(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="datetime", profile: bool = False):
    """
    Optimized pure-Python extractor. Uses local bindings and datetime.fromisoformat
    when the default format is used (faster than strptime). Returns same keys and
    the same _timings structure as the polars implementation when profile=True.
    """
    start_total = time.perf_counter_ns()

    opens, highs, lows, closes, vols = [], [], [], [], []
    times = []
    parsed_any = False

    parse_ns = 0
    conv_ns = 0
    cols_ns = 0

    # local bindings for speed
    _append_o = opens.append
    _append_h = highs.append
    _append_l = lows.append
    _append_c = closes.append
    _append_v = vols.append
    _append_t = times.append
    _fromiso = None
    use_fromiso = False

    # decide parser: prefer fromisoformat for default pattern (faster)
    if time_type != "str":
        if time_fmt == "%Y-%m-%d %H:%M:%S":
            try:
                _fromiso = datetime.fromisoformat
                # test a sample parse to ensure compatibility
                _ = _fromiso("2025-09-19 15:55:00")
                use_fromiso = True
            except Exception:
                use_fromiso = False

    t_loop_start = time.perf_counter_ns()
    if use_fromiso:
        for r in json_data:
            t_cols_start = time.perf_counter_ns()
            _append_o(r.get("open"))
            _append_h(r.get("high"))
            _append_l(r.get("low"))
            _append_c(r.get("close"))
            _append_v(r.get("volume"))
            t_cols_end = time.perf_counter_ns()
            cols_ns += (t_cols_end - t_cols_start)

            d = r.get("date")
            if d is None:
                _append_t(None)
            else:
                if time_type == "str":
                    _append_t(d)
                else:
                    t_parse_start = time.perf_counter_ns()
                    dt = _fromiso(d)
                    t_parse_end = time.perf_counter_ns()
                    parse_ns += (t_parse_end - t_parse_start)
                    parsed_any = True
                    if time_type == "timestamp":
                        t_conv_start = time.perf_counter_ns()
                        _append_t(int(dt.replace(tzinfo=None).timestamp()))
                        t_conv_end = time.perf_counter_ns()
                        conv_ns += (t_conv_end - t_conv_start)
                    else:
                        _append_t(dt)
    else:
        # fallback to strptime
        _strptime = datetime.strptime
        for r in json_data:
            t_cols_start = time.perf_counter_ns()
            _append_o(r.get("open"))
            _append_h(r.get("high"))
            _append_l(r.get("low"))
            _append_c(r.get("close"))
            _append_v(r.get("volume"))
            t_cols_end = time.perf_counter_ns()
            cols_ns += (t_cols_end - t_cols_start)

            d = r.get("date")
            if d is None:
                _append_t(None)
            else:
                if time_type == "str":
                    _append_t(d)
                else:
                    t_parse_start = time.perf_counter_ns()
                    dt = _strptime(d, time_fmt)
                    t_parse_end = time.perf_counter_ns()
                    parse_ns += (t_parse_end - t_parse_start)
                    parsed_any = True
                    if time_type == "timestamp":
                        t_conv_start = time.perf_counter_ns()
                        _append_t(int(dt.replace(tzinfo=None).timestamp()))
                        t_conv_end = time.perf_counter_ns()
                        conv_ns += (t_conv_end - t_conv_start)
                    else:
                        _append_t(dt)

    t_loop = time.perf_counter_ns() - t_loop_start
    total_ns = time.perf_counter_ns() - start_total

    result = {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}

    if profile:
        result["_timings"] = {
            "total_ms": round(total_ns / 1_000_000, 6),
            "df_construct_ms": 0.0,
            "date_parse_ms": round(parse_ns / 1_000_000, 6),
            "time_convert_ms": round(conv_ns / 1_000_000, 6),
            "cols_extract_ms": round(cols_ns / 1_000_000, 6),
            "loop_ms": round(t_loop / 1_000_000, 6),
            "parsed_date_column": parsed_any,
        }

    return result
# ...existing code...

In [115]:
# ...existing code...
def benchmark_json_to_ohlcv(json_data, iterations=50, warmup=5):
    """
    Warm-up + repeated runs for both implementations. Returns median timings for
    each profiling key and prints per-run raw timings.
    """
    import time, gc, statistics, copy, json

    # helpers to collect a flat dict of timing values per run (ms)
    def collect_timings(fn):
        runs = []
        # warmup
        for _ in range(warmup):
            _ = fn(json_data, profile=False)
        for _ in range(iterations):
            gc.collect()
            t0 = time.perf_counter_ns()
            res = fn(json_data, profile=True)
            t1 = time.perf_counter_ns()
            # ensure we capture reported timings but also measured total (in case key missing)
            reported = res.get("_timings", {})
            reported = dict(reported)  # shallow copy
            # fallback to measured total if not present
            if "total_ms" not in reported:
                reported["total_ms"] = round((t1 - t0) / 1_000_000, 6)
            runs.append(reported)
        return runs

    polars_runs = collect_timings(json_to_ohlcv_lists_polars)
    py_runs = collect_timings(json_to_ohlcv_lists_py)

    # gather keys union
    all_keys = set()
    for r in polars_runs + py_runs:
        all_keys.update(r.keys())

    def summarize(runs):
        summary = {}
        for k in sorted(all_keys):
            vals = [r.get(k, None) for r in runs]
            # drop None
            vals = [v for v in vals if v is not None]
            if not vals:
                summary[k] = None
            else:
                summary[k] = {
                    "median_ms": round(statistics.median(vals), 6),
                    "min_ms": round(min(vals), 6),
                    "max_ms": round(max(vals), 6),
                }
        return summary

    out = {
        "polars": summarize(polars_runs),
        "py": summarize(py_runs),
        "polars_raw": polars_runs,
        "py_raw": py_runs,
    }

    # print compact summary
    print("Median summary (ms):")
    keys = sorted(all_keys)
    print("\nPOLARS")
    for k in keys:
        if out["polars"].get(k) is not None:
            print(f"  {k:15}: {out['polars'][k]['median_ms']}")
    print("\nPURE PY")
    for k in keys:
        if out["py"].get(k) is not None:
            print(f"  {k:15}: {out['py'][k]['median_ms']}")
    return out

# Example usage (adjust iterations for longer runs):
bench = benchmark_json_to_ohlcv(json_data, iterations=40, warmup=4)
# ...existing code...

Median summary (ms):

POLARS
  cols_extract_ms: 0.06505
  date_parse_ms  : 0.4936
  df_construct_ms: 0.5093
  parsed_date_column: 1.0
  time_convert_ms: 0.19745
  total_ms       : 1.32405

PURE PY
  cols_extract_ms: 0.2936
  date_parse_ms  : 0.15175
  df_construct_ms: 0.0
  loop_ms        : 0.747
  parsed_date_column: 1.0
  time_convert_ms: 0.0
  total_ms       : 0.75285


The polars function seems faster above but not when i actually run it outside this benchmark function<br>
I should reserve it for large amounts of data.

In [113]:
# lets call these 2 functions to see if they work
res_polars = json_to_ohlcv_lists_polars(json_data, profile=True)

In [117]:
%timeit json_to_ohlcv_lists_polars(json_data, profile=False)

1.3 ms ± 88.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [114]:
res_polars['_timings']

{'total_ms': 2.3789,
 'df_construct_ms': 1.0453,
 'date_parse_ms': 0.9614,
 'time_convert_ms': 0.2354,
 'cols_extract_ms': 0.1308,
 'parsed_date_column': True}

In [57]:
res_py = json_to_ohlcv_lists_py(json_data, profile=True)

In [58]:
res_py['_timings']

{'total_ms': 0.9737,
 'df_construct_ms': 0.0,
 'date_parse_ms': 0.1888,
 'time_convert_ms': 0.0,
 'cols_extract_ms': 0.3885,
 'loop_ms': 0.9655,
 'parsed_date_column': True}

# Optimized column loader

In [85]:
# ...existing code...
def json_to_ohlcv_lists_py_fast(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="datetime"):
    """
    Fast pure-Python extractor for common JSON OHLCV records.
    - json_data: iterable of dict-like records with keys: date, open, high, low, close, volume
    - time_type: "datetime" -> Python datetime, "timestamp" -> int seconds, "str" -> original string
    Returns dict with keys: open, high, low, close, time, volume
    """

    opens, highs, lows, closes, vols, times = [], [], [], [], [], []
    a_o, a_h, a_l, a_c, a_v, a_t = opens.append, highs.append, lows.append, closes.append, vols.append, times.append

    # pick parser: prefer fromisoformat for the common pattern (faster), otherwise strptime
    use_fromiso = False
    _fromiso = None
    if time_type != "str" and time_fmt == "%Y-%m-%d %H:%M:%S":
        try:
            _fromiso = datetime.fromisoformat
            # quick sanity check
            _ = _fromiso("2025-09-19 15:55:00")
            use_fromiso = True
        except Exception:
            use_fromiso = False
    if not use_fromiso and time_type != "str":
        _strptime = datetime.strptime

    if use_fromiso:
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
            elif time_type == "str":
                a_t(d)
            else:
                dt = _fromiso(d)
                if time_type == "timestamp":
                    a_t(int(dt.replace(tzinfo=None).timestamp()))
                else:
                    a_t(dt)
    else:
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
            elif time_type == "str":
                a_t(d)
            else:
                dt = _strptime(d, time_fmt)
                if time_type == "timestamp":
                    a_t(int(dt.replace(tzinfo=None).timestamp()))
                else:
                    a_t(dt)

    return {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}
# ...existing code...

In [91]:
# ...existing code...
def json_to_ohlcv_lists_py_fast(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="timestamp"):
    """
    Fast pure-Python extractor.
    - For the common "%Y-%m-%d %H:%M:%S" + time_type="timestamp" path this uses manual slicing
      + calendar.timegm to avoid strptime/datetime overhead.
    - Falls back to datetime parsing only on format mismatch.
    Returns dict with keys: open, high, low, close, time, volume
    """

    opens, highs, lows, closes, vols, times = [], [], [], [], [], []
    a_o, a_h, a_l, a_c, a_v, a_t = opens.append, highs.append, lows.append, closes.append, vols.append, times.append

    fmt_default = "%Y-%m-%d %H:%M:%S"
    timegm = _calendar.timegm

    # fast path for integer unix timestamps and default format
    if time_type == "timestamp" and time_fmt == fmt_default:
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
            else:
                # manual slice parse: "YYYY-MM-DD HH:MM:SS"
                try:
                    # avoid allocations where possible; direct int conversion from slices
                    y = int(d[0:4]); mo = int(d[5:7]); day = int(d[8:10])
                    hh = int(d[11:13]); mi = int(d[14:16]); ss = int(d[17:19])
                    a_t(timegm((y, mo, day, hh, mi, ss, 0, 0, 0)))
                except Exception:
                    # fallback only when format doesn't match
                    try:
                        dt = datetime.strptime(d, time_fmt)
                        a_t(int(dt.replace(tzinfo=None).timestamp()))
                    except Exception:
                        a_t(None)
        return {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}

    # other paths (timestamp with non-default format, datetime objects, or raw strings)
    use_fromiso = False
    _fromiso = None
    if time_fmt == fmt_default:
        try:
            _fromiso = datetime.fromisoformat
            _ = _fromiso("2025-09-19 15:55:00")
            use_fromiso = True
        except Exception:
            use_fromiso = False

    if time_type == "str":
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            a_t(r.get("date"))
    elif time_type == "datetime":
        if use_fromiso:
            for r in json_data:
                a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
                d = r.get("date")
                if d is None:
                    a_t(None)
                else:
                    try:
                        a_t(_fromiso(d))
                    except Exception:
                        try:
                            a_t(datetime.strptime(d, time_fmt))
                        except Exception:
                            a_t(None)
        else:
            for r in json_data:
                a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
                d = r.get("date")
                if d is None:
                    a_t(None)
                else:
                    try:
                        a_t(datetime.strptime(d, time_fmt))
                    except Exception:
                        a_t(None)
    else:  # time_type == "timestamp" but non-default format
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
            else:
                try:
                    dt = _fromiso(d) if (use_fromiso and time_fmt == fmt_default) else datetime.strptime(d, time_fmt)
                    a_t(int(dt.replace(tzinfo=None).timestamp()))
                except Exception:
                    a_t(None)

    return {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}
# ...existing code...

## Final version of function (pure python)

In [99]:
# ...existing code...
def json_to_ohlcv_lists_py_fast(json_data, time_fmt="%Y-%m-%d %H:%M:%S", time_type="datetime"):
    """
    Fast pure-Python extractor for common JSON OHLCV records.
    - json_data: iterable of dict-like records with keys: date, open, high, low, close, volume
    - time_type: "datetime" -> Python datetime, "timestamp" -> int seconds, "str" -> original string
    Returns dict with keys: open, high, low, close, time, volume
    """

    opens, highs, lows, closes, vols, times = [], [], [], [], [], []
    a_o, a_h, a_l, a_c, a_v, a_t = opens.append, highs.append, lows.append, closes.append, vols.append, times.append

    fmt_default = "%Y-%m-%d %H:%M:%S"
    timegm = _calendar.timegm

    # Fast path: parse strings directly into epoch seconds for the common default format
    if time_type == "timestamp" and time_fmt == fmt_default:
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
                continue
            # manual slice parse: "YYYY-MM-DD HH:MM:SS"
            try:
                y = int(d[0:4]); mo = int(d[5:7]); day = int(d[8:10])
                hh = int(d[11:13]); mi = int(d[14:16]); ss = int(d[17:19])
                a_t(timegm((y, mo, day, hh, mi, ss, 0, 0, 0)))
            except Exception:
                # fallback to datetime.strptime only when format mismatch
                try:
                    dt = datetime.strptime(d, time_fmt)
                    a_t(int(dt.timestamp()))
                except Exception:
                    a_t(None)
        return {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}

    # Non-fast paths: prefer fromiso when applicable (faster than strptime)
    use_fromiso = False
    _fromiso = None
    if time_type != "str" and time_fmt == fmt_default:
        try:
            _fromiso = datetime.fromisoformat
            _ = _fromiso("2025-09-19 15:55:00")
            use_fromiso = True
        except Exception:
            use_fromiso = False

    if time_type == "str":
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            a_t(r.get("date"))
    elif time_type == "datetime":
        if use_fromiso:
            for r in json_data:
                a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
                d = r.get("date")
                if d is None:
                    a_t(None)
                else:
                    try:
                        a_t(_fromiso(d))
                    except Exception:
                        try:
                            a_t(datetime.strptime(d, time_fmt))
                        except Exception:
                            a_t(None)
        else:
            for r in json_data:
                a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
                d = r.get("date")
                if d is None:
                    a_t(None)
                else:
                    try:
                        a_t(datetime.strptime(d, time_fmt))
                    except Exception:
                        a_t(None)
    else:  # time_type == "timestamp" but non-default format
        if use_fromiso and time_fmt == fmt_default:
            parse_fn = _fromiso
        else:
            parse_fn = datetime.strptime
        for r in json_data:
            a_o(r.get("open")); a_h(r.get("high")); a_l(r.get("low")); a_c(r.get("close")); a_v(r.get("volume"))
            d = r.get("date")
            if d is None:
                a_t(None)
                continue
            try:
                dt = parse_fn(d, time_fmt) if parse_fn is datetime.strptime else parse_fn(d)
                # use dt.timestamp() (int) instead of dt.replace(...).timestamp() to avoid extra object ops
                a_t(int(dt.timestamp()))
            except Exception:
                a_t(None)

    return {"open": opens, "high": highs, "low": lows, "close": closes, "time": times, "volume": vols}
# ...existing code...

In [105]:
res_py_fast = json_to_ohlcv_lists_py_fast(json_data, time_type="timestamp")

In [110]:
res_py_fast = json_to_ohlcv_lists_py_fast(json_data, time_type="datetime")

In [109]:
type(res_py_fast['time'][0])

int

In [111]:
type(res_py_fast['time'][0])

datetime.datetime

In [98]:
%timeit json_to_ohlcv_lists_py_fast(json_data)

346 μs ± 18.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## FMP 5 Minute chart function

In [1]:
# load autoreload extension so i don't have to restart kernel on code changes
%load_ext autoreload
%autoreload 2

In [2]:
# import FMPClient from time_series_tools.py
from time_series_tools import FMPClient

In [3]:
# init the client
my_fmp = FMPClient()

In [4]:
dir(my_fmp)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_api_key',
 '_config_path',
 '_read_key_from_file',
 'api_key',
 'api_key_masked',
 'build_holidays_url',
 'clear_key',
 'fetch_holidays',
 'get_1min_chart',
 'get_5min_chart']

In [14]:
data=my_fmp.get_5min_chart("PLTR", to_date="2025-09-08", from_date="2025-09-09")
data[0]

[{'date': '2025-09-08 15:55:00',
  'open': 156.49,
  'low': 155.83,
  'high': 156.5,
  'close': 156.1,
  'volume': 1491436},
 {'date': '2025-09-08 15:50:00',
  'open': 156.31,
  'low': 156.1,
  'high': 156.54,
  'close': 156.48,
  'volume': 713837},
 {'date': '2025-09-08 15:45:00',
  'open': 156.31,
  'low': 156.29,
  'high': 156.45,
  'close': 156.33,
  'volume': 329125},
 {'date': '2025-09-08 15:40:00',
  'open': 156.15,
  'low': 156.12,
  'high': 156.48,
  'close': 156.31,
  'volume': 371065},
 {'date': '2025-09-08 15:35:00',
  'open': 156.04,
  'low': 156,
  'high': 156.25,
  'close': 156.13,
  'volume': 311626},
 {'date': '2025-09-08 15:30:00',
  'open': 156.09,
  'low': 156.04,
  'high': 156.38,
  'close': 156.04,
  'volume': 379499},
 {'date': '2025-09-08 15:25:00',
  'open': 156.01,
  'low': 155.95,
  'high': 156.22,
  'close': 156.08,
  'volume': 406577},
 {'date': '2025-09-08 15:20:00',
  'open': 155.9,
  'low': 155.77,
  'high': 156.02,
  'close': 156.01,
  'volume': 425795}

In [8]:
# get the kwargs for the function
import inspect
inspect.signature(my_fmp.get_5min_chart)
# print the names of the kwargs
inspect.signature(my_fmp.get_5min_chart).parameters.keys()

odict_keys(['symbol', 'from_date', 'to_date', 'nonadjusted', 'timeout'])

In [6]:
# test the 1min chart
data_1min=my_fmp.get_1min_chart("PLTR", to_date="2025-09-26", from_date="2025-09-25")
data_1min[0]

HTTPError: 402 Client Error: Payment Required for url: https://financialmodelingprep.com/stable/historical-chart/1min?symbol=PLTR&apikey=de1aeb80bdc8cd954883fe633c30af89&from=2025-09-25&to=2025-09-26